In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

from tqdm.auto import tqdm

import copy
import time

c:\Users\David\Documents\4530\ACIT4530\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1 Load Step_1 results

We load the filtered interaction data and the pairwise datasets created in Step 1.

The interaction-level splits are retained for later individual and group ranking evaluation.  
The pairwise datasets are used in this step for BPR training and pairwise prediction evaluation.

In [2]:
data_dir = "./results_step1"

required_files = [
    "ratings_filtered.parquet",
    "train_pairs.parquet",
    "validation_pairs.parquet",
    "test_pairs.parquet",
    "step1_config.json",
    "train_split_model_universe.parquet",
    "validation_split_model_universe.parquet",
    "test_split_model_universe.parquet",
]

missing_files = [
    filename
    for filename in required_files
    if not os.path.exists(f"{data_dir}/{filename}")
]

if missing_files:
    raise FileNotFoundError(
        f"Missing Step 1 output files in {data_dir}: {missing_files}"
    )


# Main filtered interaction data
ratings_filtered = pd.read_parquet(
    f"{data_dir}/ratings_filtered.parquet"
)


# For Step 2 ranking evaluation, use model-universe interaction splits.
# These were created in Step 1 so that users/items match the final train pair universe.
train_df = pd.read_parquet(
    f"{data_dir}/train_split_model_universe.parquet"
)

val_df = pd.read_parquet(
    f"{data_dir}/validation_split_model_universe.parquet"
)

test_df = pd.read_parquet(
    f"{data_dir}/test_split_model_universe.parquet"
)


# Pairwise datasets for BPR and pairwise evaluation
train_pairs = pd.read_parquet(
    f"{data_dir}/train_pairs.parquet"
)

val_pairs = pd.read_parquet(
    f"{data_dir}/validation_pairs.parquet"
)

test_pairs = pd.read_parquet(
    f"{data_dir}/test_pairs.parquet"
)

In [3]:
with open(f"{data_dir}/step1_config.json", "r") as f:
    step1_config = json.load(f)


# Robust config reading:
# works with both old and improved Step 1 configs.
random_seed = int(
    step1_config.get("random_seed", step1_config.get("seed", 42))
)

pair_min_diff = int(
    step1_config.get("pair_min_diff", 2)
)

min_user_ratings_pre_split = int(
    step1_config.get(
        "min_user_ratings_pre_split",
        step1_config.get("min_user_ratings", -1),
    )
)

min_movie_ratings_pre_split = int(
    step1_config.get(
        "min_movie_ratings_pre_split",
        step1_config.get("min_movie_ratings", -1),
    )
)


print("Loaded Step 1 outputs from:", data_dir)
print("random_seed:", random_seed)
print("pair_min_diff:", pair_min_diff)
print("min_user_ratings_pre_split:", min_user_ratings_pre_split)
print("min_movie_ratings_pre_split:", min_movie_ratings_pre_split)

Loaded Step 1 outputs from: ./results_step1
random_seed: 42
pair_min_diff: 2
min_user_ratings_pre_split: 40
min_movie_ratings_pre_split: 10


In [4]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Random seed:", random_seed)

Device: cpu
Random seed: 42


### 1.1 Standardize Pairwise Column Names

Step 1 may save pairwise columns using either descriptive names:

- `preferred_item`
- `less_preferred_item`
- `rating_preferred`
- `rating_less_preferred`

or BPR-style names:

- `pos_item`
- `neg_item`
- `rating_pos`
- `rating_neg`

For Step 2, we standardize all pairwise datasets to the BPR-style names.

In [5]:
def standardize_pair_columns(pair_df):
    df = pair_df.copy()

    rename_map = {
        "preferred_item": "pos_item",
        "less_preferred_item": "neg_item",
        "rating_preferred": "rating_pos",
        "rating_less_preferred": "rating_neg",
        "timestamp_preferred": "timestamp_pos",
        "timestamp_less_preferred": "timestamp_neg",
    }

    active_rename_map = {
        old_name: new_name
        for old_name, new_name in rename_map.items()
        if old_name in df.columns
    }

    df = df.rename(columns=active_rename_map)

    required_columns = [
        "user_id",
        "pos_item",
        "neg_item",
        "rating_pos",
        "rating_neg",
        "rating_diff",
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required pairwise columns after standardization: {missing_columns}"
        )

    return df


train_pairs = standardize_pair_columns(train_pairs)
val_pairs = standardize_pair_columns(val_pairs)
test_pairs = standardize_pair_columns(test_pairs)

display(train_pairs.head())
print("Train pair columns:", train_pairs.columns.tolist())

,user_id,pos_item,neg_item,rating_pos,rating_neg,rating_diff,timestamp_pos,timestamp_neg
0,1,1022,2340,5,3,2,978300055,978300103
1,1,1270,2340,5,3,2,978300055,978300103
2,1,1022,720,5,3,2,978300055,978300760
3,1,1270,720,5,3,2,978300055,978300760
4,1,1022,914,5,3,2,978300055,978301968


Train pair columns: ['user_id', 'pos_item', 'neg_item', 'rating_pos', 'rating_neg', 'rating_diff', 'timestamp_pos', 'timestamp_neg']


### 1.2 Load summary

In [6]:
load_summary = pd.DataFrame([
    {
        "dataset": "ratings_filtered",
        "rows": len(ratings_filtered),
        "columns": ratings_filtered.shape[1],
        "users": ratings_filtered["user_id"].nunique(),
        "items": ratings_filtered["movie_id"].nunique(),
    },
    {
        "dataset": "train_df_model_universe",
        "rows": len(train_df),
        "columns": train_df.shape[1],
        "users": train_df["user_id"].nunique(),
        "items": train_df["movie_id"].nunique(),
    },
    {
        "dataset": "val_df_model_universe",
        "rows": len(val_df),
        "columns": val_df.shape[1],
        "users": val_df["user_id"].nunique(),
        "items": val_df["movie_id"].nunique(),
    },
    {
        "dataset": "test_df_model_universe",
        "rows": len(test_df),
        "columns": test_df.shape[1],
        "users": test_df["user_id"].nunique(),
        "items": test_df["movie_id"].nunique(),
    },
    {
        "dataset": "train_pairs",
        "rows": len(train_pairs),
        "columns": train_pairs.shape[1],
        "users": train_pairs["user_id"].nunique(),
        "items": len(
            set(train_pairs["pos_item"].unique())
            | set(train_pairs["neg_item"].unique())
        ),
    },
    {
        "dataset": "val_pairs",
        "rows": len(val_pairs),
        "columns": val_pairs.shape[1],
        "users": val_pairs["user_id"].nunique(),
        "items": len(
            set(val_pairs["pos_item"].unique())
            | set(val_pairs["neg_item"].unique())
        ),
    },
    {
        "dataset": "test_pairs",
        "rows": len(test_pairs),
        "columns": test_pairs.shape[1],
        "users": test_pairs["user_id"].nunique(),
        "items": len(
            set(test_pairs["pos_item"].unique())
            | set(test_pairs["neg_item"].unique())
        ),
    },
])

display(load_summary)

,dataset,rows,columns,users,items
0,ratings_filtered,960916,5,4726,3250
1,train_df_model_universe,759306,5,4722,3250
2,val_df_model_universe,100787,5,4722,3204
3,test_df_model_universe,100564,5,4722,3226
4,train_pairs,5692244,8,4722,3250
5,val_pairs,478098,8,4158,3193
6,test_pairs,507550,8,4164,3222


### 1.3 Pairwise Sanity Checks

Before training, we verify that the loaded pairwise datasets are valid.

In [7]:
def semantic_duplicate_count(pair_df):
    return pair_df.duplicated(
        ["user_id", "pos_item", "neg_item"]
    ).sum()


def reverse_conflict_count(pair_df):
    forward = set(
        zip(
            pair_df["user_id"],
            pair_df["pos_item"],
            pair_df["neg_item"],
        )
    )

    reverse = set(
        zip(
            pair_df["user_id"],
            pair_df["neg_item"],
            pair_df["pos_item"],
        )
    )

    return len(forward & reverse)


pair_quality_rows = []

for split_name, pair_df in [
    ("train", train_pairs),
    ("validation", val_pairs),
    ("test", test_pairs),
]:
    pair_quality_rows.append({
        "split": split_name,
        "pairs": len(pair_df),
        "users": pair_df["user_id"].nunique(),
        "items": len(
            set(pair_df["pos_item"].unique())
            | set(pair_df["neg_item"].unique())
        ),
        "min_rating_diff": int(pair_df["rating_diff"].min()),
        "all_rating_diff_ge_min": bool(
            pair_df["rating_diff"].ge(pair_min_diff).all()
        ),
        "semantic_duplicates": int(
            semantic_duplicate_count(pair_df)
        ),
        "reverse_conflicts": int(
            reverse_conflict_count(pair_df)
        ),
        "same_pos_neg_items": int(
            (pair_df["pos_item"] == pair_df["neg_item"]).sum()
        ),
    })

pair_quality_summary = pd.DataFrame(pair_quality_rows)
display(pair_quality_summary)


assert pair_quality_summary["all_rating_diff_ge_min"].all()
assert pair_quality_summary["semantic_duplicates"].sum() == 0
assert pair_quality_summary["reverse_conflicts"].sum() == 0
assert pair_quality_summary["same_pos_neg_items"].sum() == 0

,split,pairs,users,items,min_rating_diff,all_rating_diff_ge_min,semantic_duplicates,reverse_conflicts,same_pos_neg_items
0,train,5692244,4722,3250,2,True,0,0,0
1,validation,478098,4158,3193,2,True,0,0,0
2,test,507550,4164,3222,2,True,0,0,0


### 1.4 Universe checks

In [8]:
train_pair_users = set(train_pairs["user_id"].unique())
train_pair_items = (
    set(train_pairs["pos_item"].unique())
    | set(train_pairs["neg_item"].unique())
)

val_pair_users = set(val_pairs["user_id"].unique())
val_pair_items = (
    set(val_pairs["pos_item"].unique())
    | set(val_pairs["neg_item"].unique())
)

test_pair_users = set(test_pairs["user_id"].unique())
test_pair_items = (
    set(test_pairs["pos_item"].unique())
    | set(test_pairs["neg_item"].unique())
)


train_model_users = set(train_df["user_id"].unique())
train_model_items = set(train_df["movie_id"].unique())

val_model_users = set(val_df["user_id"].unique())
val_model_items = set(val_df["movie_id"].unique())

test_model_users = set(test_df["user_id"].unique())
test_model_items = set(test_df["movie_id"].unique())


universe_check_summary = pd.DataFrame([
    {
        "check": "train pair users == train model users",
        "result": train_pair_users == train_model_users,
        "left_count": len(train_pair_users),
        "right_count": len(train_model_users),
        "difference_count": len(train_pair_users.symmetric_difference(train_model_users)),
    },
    {
        "check": "train pair items subset of train model items",
        "result": train_pair_items.issubset(train_model_items),
        "left_count": len(train_pair_items),
        "right_count": len(train_model_items),
        "difference_count": len(train_pair_items - train_model_items),
    },
    {
        "check": "validation pair users subset of train pair users",
        "result": val_pair_users.issubset(train_pair_users),
        "left_count": len(val_pair_users),
        "right_count": len(train_pair_users),
        "difference_count": len(val_pair_users - train_pair_users),
    },
    {
        "check": "test pair users subset of train pair users",
        "result": test_pair_users.issubset(train_pair_users),
        "left_count": len(test_pair_users),
        "right_count": len(train_pair_users),
        "difference_count": len(test_pair_users - train_pair_users),
    },
    {
        "check": "validation pair items subset of train pair items",
        "result": val_pair_items.issubset(train_pair_items),
        "left_count": len(val_pair_items),
        "right_count": len(train_pair_items),
        "difference_count": len(val_pair_items - train_pair_items),
    },
    {
        "check": "test pair items subset of train pair items",
        "result": test_pair_items.issubset(train_pair_items),
        "left_count": len(test_pair_items),
        "right_count": len(train_pair_items),
        "difference_count": len(test_pair_items - train_pair_items),
    },
    {
        "check": "validation model users subset of train pair users",
        "result": val_model_users.issubset(train_pair_users),
        "left_count": len(val_model_users),
        "right_count": len(train_pair_users),
        "difference_count": len(val_model_users - train_pair_users),
    },
    {
        "check": "test model users subset of train pair users",
        "result": test_model_users.issubset(train_pair_users),
        "left_count": len(test_model_users),
        "right_count": len(train_pair_users),
        "difference_count": len(test_model_users - train_pair_users),
    },
    {
        "check": "validation model items subset of train pair items",
        "result": val_model_items.issubset(train_pair_items),
        "left_count": len(val_model_items),
        "right_count": len(train_pair_items),
        "difference_count": len(val_model_items - train_pair_items),
    },
    {
        "check": "test model items subset of train pair items",
        "result": test_model_items.issubset(train_pair_items),
        "left_count": len(test_model_items),
        "right_count": len(train_pair_items),
        "difference_count": len(test_model_items - train_pair_items),
    },
])

display(universe_check_summary)

assert universe_check_summary["result"].all()

,check,result,left_count,right_count,difference_count
0,train pair users == train model users,True,4722,4722,0
1,train pair items subset of train model items,True,3250,3250,0
2,validation pair users subset of train pair users,True,4158,4722,0
3,test pair users subset of train pair users,True,4164,4722,0
4,validation pair items subset of train pair items,True,3193,3250,0
5,test pair items subset of train pair items,True,3222,3250,0
6,validation model users subset of train pair users,True,4722,4722,0
7,test model users subset of train pair users,True,4722,4722,0
8,validation model items subset of train pair items,True,3204,3250,0
9,test model items subset of train pair items,True,3226,3250,0


# 2 Create User and Item Index Mappings

- We build compact integer mappings only from the train pair universe.
- This is the actual universe learned by the BPR model.

In [9]:
train_users = sorted(train_pairs["user_id"].unique())

train_items = sorted(
    set(train_pairs["pos_item"].unique())
    | set(train_pairs["neg_item"].unique())
)

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(train_users)
}

item_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(train_items)
}

idx_to_user = {
    idx: user_id
    for user_id, idx in user_to_idx.items()
}

idx_to_item = {
    idx: movie_id
    for movie_id, idx in item_to_idx.items()
}

n_users = len(user_to_idx)
n_items = len(item_to_idx)

mapping_summary = pd.DataFrame([
    {
        "entity": "users",
        "count": n_users,
        "min_original_id": min(train_users),
        "max_original_id": max(train_users),
        "min_index": 0,
        "max_index": n_users - 1,
    },
    {
        "entity": "items",
        "count": n_items,
        "min_original_id": min(train_items),
        "max_original_id": max(train_items),
        "min_index": 0,
        "max_index": n_items - 1,
    },
])

display(mapping_summary)

,entity,count,min_original_id,max_original_id,min_index,max_index
0,users,4722,1,6040,0,4721
1,items,3250,1,3952,0,3249


### 2.1 Save mapping tables

In [10]:
user_index_table = (
    pd.DataFrame({
        "user_idx": list(idx_to_user.keys()),
        "user_id": list(idx_to_user.values()),
    })
    .sort_values("user_idx")
    .reset_index(drop=True)
)

item_index_table = (
    pd.DataFrame({
        "item_idx": list(idx_to_item.keys()),
        "movie_id": list(idx_to_item.values()),
    })
    .sort_values("item_idx")
    .reset_index(drop=True)
)

display(user_index_table.head())
display(item_index_table.head())

assert user_index_table["user_idx"].is_unique
assert user_index_table["user_id"].is_unique
assert item_index_table["item_idx"].is_unique
assert item_index_table["movie_id"].is_unique

assert user_index_table["user_idx"].min() == 0
assert user_index_table["user_idx"].max() == n_users - 1

assert item_index_table["item_idx"].min() == 0
assert item_index_table["item_idx"].max() == n_items - 1

,user_idx,user_id
0,0,1
1,1,2
2,2,3
3,3,5
4,4,6


,item_idx,movie_id
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5


### 2.2 Add indices to pairwise datasets

In [11]:
def add_pair_indices(pair_df, user_to_idx, item_to_idx, split_name):
    df = pair_df.copy()

    before_rows = len(df)

    df["user_idx"] = df["user_id"].map(user_to_idx)
    df["pos_item_idx"] = df["pos_item"].map(item_to_idx)
    df["neg_item_idx"] = df["neg_item"].map(item_to_idx)

    missing_user_rows = int(df["user_idx"].isna().sum())
    missing_pos_item_rows = int(df["pos_item_idx"].isna().sum())
    missing_neg_item_rows = int(df["neg_item_idx"].isna().sum())

    if missing_user_rows > 0 or missing_pos_item_rows > 0 or missing_neg_item_rows > 0:
        raise ValueError(
            f"{split_name}: missing index mappings. "
            f"missing_user_rows={missing_user_rows}, "
            f"missing_pos_item_rows={missing_pos_item_rows}, "
            f"missing_neg_item_rows={missing_neg_item_rows}"
        )

    df["user_idx"] = df["user_idx"].astype("int64")
    df["pos_item_idx"] = df["pos_item_idx"].astype("int64")
    df["neg_item_idx"] = df["neg_item_idx"].astype("int64")

    after_rows = len(df)

    summary = {
        "split": split_name,
        "rows_before": before_rows,
        "rows_after": after_rows,
        "rows_removed": before_rows - after_rows,
        "missing_user_rows": missing_user_rows,
        "missing_pos_item_rows": missing_pos_item_rows,
        "missing_neg_item_rows": missing_neg_item_rows,
        "min_user_idx": int(df["user_idx"].min()),
        "max_user_idx": int(df["user_idx"].max()),
        "min_item_idx": int(
            min(df["pos_item_idx"].min(), df["neg_item_idx"].min())
        ),
        "max_item_idx": int(
            max(df["pos_item_idx"].max(), df["neg_item_idx"].max())
        ),
        "same_pos_neg_idx": int(
            (df["pos_item_idx"] == df["neg_item_idx"]).sum()
        ),
    }

    return df, summary


train_pairs_idx, train_index_summary = add_pair_indices(
    train_pairs,
    user_to_idx,
    item_to_idx,
    "train",
)

val_pairs_idx, val_index_summary = add_pair_indices(
    val_pairs,
    user_to_idx,
    item_to_idx,
    "validation",
)

test_pairs_idx, test_index_summary = add_pair_indices(
    test_pairs,
    user_to_idx,
    item_to_idx,
    "test",
)

index_summary = pd.DataFrame([
    train_index_summary,
    val_index_summary,
    test_index_summary,
])

display(index_summary)
display(train_pairs_idx.head())

,split,rows_before,rows_after,rows_removed,missing_user_rows,missing_pos_item_rows,missing_neg_item_rows,min_user_idx,max_user_idx,min_item_idx,max_item_idx,same_pos_neg_idx
0,train,5692244,5692244,0,0,0,0,0,4721,0,3249,0
1,validation,478098,478098,0,0,0,0,0,4721,0,3249,0
2,test,507550,507550,0,0,0,0,1,4721,0,3249,0


,user_id,pos_item,neg_item,rating_pos,rating_neg,rating_diff,timestamp_pos,timestamp_neg,user_idx,pos_item_idx,neg_item_idx
0,1,1022,2340,5,3,2,978300055,978300103,0,821,1880
1,1,1270,2340,5,3,2,978300055,978300103,0,1020,1880
2,1,1022,720,5,3,2,978300055,978300760,0,821,609
3,1,1270,720,5,3,2,978300055,978300760,0,1020,609
4,1,1022,914,5,3,2,978300055,978301968,0,821,727


### 2.3 Indexed data checks

In [12]:
assert index_summary["rows_removed"].sum() == 0

assert index_summary["min_user_idx"].min() >= 0
assert index_summary["max_user_idx"].max() < n_users

assert index_summary["min_item_idx"].min() >= 0
assert index_summary["max_item_idx"].max() < n_items

assert index_summary["same_pos_neg_idx"].sum() == 0

for split_name, pair_df in [
    ("train", train_pairs_idx),
    ("validation", val_pairs_idx),
    ("test", test_pairs_idx),
]:
    assert pair_df["user_idx"].between(0, n_users - 1).all(), (
        f"{split_name}: invalid user_idx"
    )

    assert pair_df["pos_item_idx"].between(0, n_items - 1).all(), (
        f"{split_name}: invalid pos_item_idx"
    )

    assert pair_df["neg_item_idx"].between(0, n_items - 1).all(), (
        f"{split_name}: invalid neg_item_idx"
    )

    assert (pair_df["pos_item_idx"] != pair_df["neg_item_idx"]).all(), (
        f"{split_name}: found same positive and negative item index"
    )

print("Indexed pairwise data checks passed.")

Indexed pairwise data checks passed.


### 3 PyTorch Dataset + DataLoader

In [13]:
class PairwisePreferenceDataset(Dataset):
    def __init__(self, pair_df):
        self.users = torch.tensor(
            pair_df["user_idx"].values,
            dtype=torch.long,
        )

        self.pos_items = torch.tensor(
            pair_df["pos_item_idx"].values,
            dtype=torch.long,
        )

        self.neg_items = torch.tensor(
            pair_df["neg_item_idx"].values,
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return (
            self.users[idx],
            self.pos_items[idx],
            self.neg_items[idx],
        )


train_dataset = PairwisePreferenceDataset(train_pairs_idx)
val_dataset = PairwisePreferenceDataset(val_pairs_idx)
test_dataset = PairwisePreferenceDataset(test_pairs_idx)

dataset_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_dataset),
    },
    {
        "split": "validation",
        "rows": len(val_dataset),
    },
    {
        "split": "test",
        "rows": len(test_dataset),
    },
])

display(dataset_summary)

,split,rows
0,train,5692244
1,validation,478098
2,test,507550


In [14]:
dataset_check_rows = []

for split_name, dataset in [
    ("train", train_dataset),
    ("validation", val_dataset),
    ("test", test_dataset),
]:
    users = dataset.users
    pos_items = dataset.pos_items
    neg_items = dataset.neg_items

    dataset_check_rows.append({
        "split": split_name,
        "rows": len(dataset),
        "user_dtype": str(users.dtype),
        "pos_item_dtype": str(pos_items.dtype),
        "neg_item_dtype": str(neg_items.dtype),
        "min_user_idx": int(users.min()),
        "max_user_idx": int(users.max()),
        "min_pos_item_idx": int(pos_items.min()),
        "max_pos_item_idx": int(pos_items.max()),
        "min_neg_item_idx": int(neg_items.min()),
        "max_neg_item_idx": int(neg_items.max()),
        "same_pos_neg_items": int((pos_items == neg_items).sum()),
    })

dataset_check_summary = pd.DataFrame(dataset_check_rows)
display(dataset_check_summary)

assert dataset_check_summary["min_user_idx"].min() >= 0
assert dataset_check_summary["max_user_idx"].max() < n_users

assert dataset_check_summary["min_pos_item_idx"].min() >= 0
assert dataset_check_summary["max_pos_item_idx"].max() < n_items

assert dataset_check_summary["min_neg_item_idx"].min() >= 0
assert dataset_check_summary["max_neg_item_idx"].max() < n_items

assert dataset_check_summary["same_pos_neg_items"].sum() == 0

,split,rows,user_dtype,pos_item_dtype,neg_item_dtype,min_user_idx,max_user_idx,min_pos_item_idx,max_pos_item_idx,min_neg_item_idx,max_neg_item_idx,same_pos_neg_items
0,train,5692244,torch.int64,torch.int64,torch.int64,0,4721,0,3249,0,3249,0
1,validation,478098,torch.int64,torch.int64,torch.int64,0,4721,0,3249,0,3249,0
2,test,507550,torch.int64,torch.int64,torch.int64,1,4721,0,3249,0,3249,0


# 4 DataLoader Configuration

In [15]:
def make_pairwise_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size,
    seed,
):
    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=train_generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    loader_summary = pd.DataFrame([
        {
            "split": "train",
            "rows": len(train_dataset),
            "batch_size": batch_size,
            "n_batches": len(train_loader),
            "shuffle": True,
        },
        {
            "split": "validation",
            "rows": len(val_dataset),
            "batch_size": batch_size,
            "n_batches": len(val_loader),
            "shuffle": False,
        },
        {
            "split": "test",
            "rows": len(test_dataset),
            "batch_size": batch_size,
            "n_batches": len(test_loader),
            "shuffle": False,
        },
    ])

    return train_loader, val_loader, test_loader, loader_summary

In [16]:
baseline_batch_size = 16_384

train_loader, val_loader, test_loader, loader_summary = make_pairwise_loaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=baseline_batch_size,
    seed=random_seed,
)

display(loader_summary)

,split,rows,batch_size,n_batches,shuffle
0,train,5692244,16384,348,True
1,validation,478098,16384,30,False
2,test,507550,16384,31,False


### 4.1 Batch Sanity Check

In [17]:
sample_users, sample_pos_items, sample_neg_items = next(iter(train_loader))

batch_check = pd.DataFrame([
    {
        "tensor": "users",
        "shape": tuple(sample_users.shape),
        "dtype": str(sample_users.dtype),
        "min_value": int(sample_users.min()),
        "max_value": int(sample_users.max()),
    },
    {
        "tensor": "pos_items",
        "shape": tuple(sample_pos_items.shape),
        "dtype": str(sample_pos_items.dtype),
        "min_value": int(sample_pos_items.min()),
        "max_value": int(sample_pos_items.max()),
    },
    {
        "tensor": "neg_items",
        "shape": tuple(sample_neg_items.shape),
        "dtype": str(sample_neg_items.dtype),
        "min_value": int(sample_neg_items.min()),
        "max_value": int(sample_neg_items.max()),
    },
])

display(batch_check)

assert sample_users.min() >= 0
assert sample_users.max() < n_users

assert sample_pos_items.min() >= 0
assert sample_pos_items.max() < n_items

assert sample_neg_items.min() >= 0
assert sample_neg_items.max() < n_items

assert (sample_pos_items != sample_neg_items).all()

,tensor,shape,dtype,min_value,max_value
0,users,"(16384,)",torch.int64,1,4721
1,pos_items,"(16384,)",torch.int64,0,3249
2,neg_items,"(16384,)",torch.int64,0,3249


# 5 BPR Matrix Factorization Model

In [18]:
class BPRMatrixFactorization(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embedding_dim=32,
        init_std=0.01,
        use_item_bias=True,
    ):
        super().__init__()

        self.n_users = n_users
        self.n_items = n_items
        self.embedding_dim = embedding_dim
        self.use_item_bias = use_item_bias

        self.user_embedding = nn.Embedding(
            n_users,
            embedding_dim,
        )

        self.item_embedding = nn.Embedding(
            n_items,
            embedding_dim,
        )

        if use_item_bias:
            self.item_bias = nn.Embedding(
                n_items,
                1,
            )
        else:
            self.item_bias = None

        self.reset_parameters(init_std)

    def reset_parameters(self, init_std):
        nn.init.normal_(
            self.user_embedding.weight,
            mean=0.0,
            std=init_std,
        )

        nn.init.normal_(
            self.item_embedding.weight,
            mean=0.0,
            std=init_std,
        )

        if self.item_bias is not None:
            nn.init.zeros_(self.item_bias.weight)

    def score(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)

        score = (user_vec * item_vec).sum(dim=1)

        if self.item_bias is not None:
            score = score + self.item_bias(item_idx).squeeze(-1)

        return score

    def forward(self, user_idx, pos_item_idx, neg_item_idx):
        pos_score = self.score(user_idx, pos_item_idx)
        neg_score = self.score(user_idx, neg_item_idx)

        return pos_score - neg_score

    def l2_regularization(self, user_idx, pos_item_idx, neg_item_idx):
        user_vec = self.user_embedding(user_idx)
        pos_vec = self.item_embedding(pos_item_idx)
        neg_vec = self.item_embedding(neg_item_idx)

        l2 = (
            user_vec.pow(2).sum(dim=1)
            + pos_vec.pow(2).sum(dim=1)
            + neg_vec.pow(2).sum(dim=1)
        ).mean()

        return l2

In [19]:
baseline_embedding_dim = 32
baseline_init_std = 0.01
baseline_use_item_bias = True

model_sanity = BPRMatrixFactorization(
    n_users=n_users,
    n_items=n_items,
    embedding_dim=baseline_embedding_dim,
    init_std=baseline_init_std,
    use_item_bias=baseline_use_item_bias,
).to(device)

model_summary = pd.DataFrame([
    {
        "component": "user_embedding",
        "shape": tuple(model_sanity.user_embedding.weight.shape),
        "parameters": model_sanity.user_embedding.weight.numel(),
    },
    {
        "component": "item_embedding",
        "shape": tuple(model_sanity.item_embedding.weight.shape),
        "parameters": model_sanity.item_embedding.weight.numel(),
    },
    {
        "component": "item_bias",
        "shape": (
            tuple(model_sanity.item_bias.weight.shape)
            if model_sanity.item_bias is not None
            else None
        ),
        "parameters": (
            model_sanity.item_bias.weight.numel()
            if model_sanity.item_bias is not None
            else 0
        ),
    },
])

model_summary["total_parameters"] = model_summary["parameters"].sum()

display(model_summary)

,component,shape,parameters,total_parameters
0,user_embedding,"(4722, 32)",151104,258354
1,item_embedding,"(3250, 32)",104000,258354
2,item_bias,"(3250, 1)",3250,258354


### Forward pass sanity check

In [20]:
sample_users = sample_users.to(device)
sample_pos_items = sample_pos_items.to(device)
sample_neg_items = sample_neg_items.to(device)

with torch.no_grad():
    sample_score_diff = model_sanity(
        sample_users,
        sample_pos_items,
        sample_neg_items,
    )

forward_check = pd.DataFrame([
    {
        "tensor": "sample_score_diff",
        "shape": tuple(sample_score_diff.shape),
        "dtype": str(sample_score_diff.dtype),
        "min_value": float(sample_score_diff.min().cpu()),
        "max_value": float(sample_score_diff.max().cpu()),
        "mean_value": float(sample_score_diff.mean().cpu()),
    }
])

display(forward_check)

assert sample_score_diff.shape[0] == sample_users.shape[0]
assert torch.isfinite(sample_score_diff).all()

,tensor,shape,dtype,min_value,max_value,mean_value
0,sample_score_diff,"(16384,)",torch.float32,-0.003651,0.003668,-0.000006


# 6 BPR Loss + Pairwise Evaluation

### 6.1 BPR loss

In [21]:
def bpr_loss(score_diff, l2_penalty=None, reg_lambda=0.0):
    """
    BPR objective:

        -log sigmoid(score(user, positive_item) - score(user, negative_item))

    Higher score_diff means the model ranks the preferred item above the less-preferred item.
    """
    ranking_loss = -F.logsigmoid(score_diff).mean()

    if l2_penalty is not None and reg_lambda > 0:
        return ranking_loss + reg_lambda * l2_penalty

    return ranking_loss

### 6.2 Evaluation helpers

In [22]:
@torch.no_grad()
def predict_pair_score_diffs(
    model,
    pair_df,
    device,
    batch_size=65_536,
):
    model.eval()

    score_diffs = []

    for start in range(0, len(pair_df), batch_size):
        batch = pair_df.iloc[start:start + batch_size]

        user_idx = torch.tensor(
            batch["user_idx"].values,
            dtype=torch.long,
            device=device,
        )

        pos_item_idx = torch.tensor(
            batch["pos_item_idx"].values,
            dtype=torch.long,
            device=device,
        )

        neg_item_idx = torch.tensor(
            batch["neg_item_idx"].values,
            dtype=torch.long,
            device=device,
        )

        batch_score_diff = model(
            user_idx,
            pos_item_idx,
            neg_item_idx,
        )

        score_diffs.append(
            batch_score_diff.detach().cpu().numpy()
        )

    return np.concatenate(score_diffs)

### 6.3 Pairwise metrics

In [23]:
def evaluate_pairwise(
    model,
    pair_df,
    device,
    split_name,
    batch_size=65_536,
):
    score_diff = predict_pair_score_diffs(
        model=model,
        pair_df=pair_df,
        device=device,
        batch_size=batch_size,
    )

    # Each row is a known directed preference:
    # pos_item should score higher than neg_item.
    correct = score_diff > 0

    comparison_accuracy_micro = float(correct.mean())

    # Macro user accuracy prevents high-pair-count users from dominating the metric.
    user_metric_df = pair_df[["user_id"]].copy()
    user_metric_df["correct"] = correct

    user_accuracy = (
        user_metric_df
        .groupby("user_id")["correct"]
        .mean()
    )

    comparison_accuracy_macro_user = float(user_accuracy.mean())

    # Symmetric AUC:
    # observed preference direction is positive;
    # reversed direction is treated as negative.
    y_true = np.concatenate([
        np.ones_like(score_diff),
        np.zeros_like(score_diff),
    ])

    y_score = np.concatenate([
        score_diff,
        -score_diff,
    ])

    roc_auc_symmetric = float(
        roc_auc_score(y_true, y_score)
    )

    probs = 1 / (1 + np.exp(-score_diff))
    probs_clipped = np.clip(probs, 1e-12, 1 - 1e-12)


    return {
    "split": split_name,
    "pairs": int(len(pair_df)),
    "users": int(pair_df["user_id"].nunique()),
    "comparison_accuracy_micro": comparison_accuracy_micro,
    "comparison_accuracy_macro_user": comparison_accuracy_macro_user,
    "roc_auc_symmetric": roc_auc_symmetric,
    "brier_positive_pairs": float(
        brier_score_loss(np.ones_like(probs), probs)
    ),
    "log_loss_positive_pairs": float(
        log_loss(np.ones_like(probs), probs_clipped, labels=[0, 1])
    ),
    "mean_score_diff": float(score_diff.mean()),
    "median_score_diff": float(np.median(score_diff)),
}

### 6.4 Evaluation sanity check before training

In [24]:
pretrain_pairwise_eval = pd.DataFrame([
    evaluate_pairwise(
        model_sanity,
        train_pairs_idx,
        device,
        "train_pretrain",
    ),
    evaluate_pairwise(
        model_sanity,
        val_pairs_idx,
        device,
        "validation_pretrain",
    ),
    evaluate_pairwise(
        model_sanity,
        test_pairs_idx,
        device,
        "test_pretrain",
    ),
])

display(pretrain_pairwise_eval)

,split,pairs,users,comparison_accuracy_micro,comparison_accuracy_macro_user,roc_auc_symmetric,brier_positive_pairs,log_loss_positive_pairs,mean_score_diff,median_score_diff
0,train_pretrain,5692244,4722,0.499598,0.500393,0.499118,0.250000,0.693148,-0.000001,-7.995623e-07
1,validation_pretrain,478098,4158,0.501447,0.504453,0.502140,0.249999,0.693145,0.000004,2.757035e-06
2,test_pretrain,507550,4164,0.498920,0.504242,0.497665,0.250001,0.693149,-0.000004,-2.073393e-06


# 7 Training

### 7.1 Train one epoch

In [25]:
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
    reg_lambda=0.0,
    show_progress=True,
):
    model.train()

    total_loss = 0.0
    total_ranking_loss = 0.0
    total_l2_penalty = 0.0
    total_examples = 0

    iterator = train_loader

    if show_progress:
        iterator = tqdm(
            train_loader,
            desc="Training batches",
            leave=False,
        )

    for users, pos_items, neg_items in iterator:
        users = users.to(device)
        pos_items = pos_items.to(device)
        neg_items = neg_items.to(device)

        optimizer.zero_grad()

        score_diff = model(
            users,
            pos_items,
            neg_items,
        )

        ranking_loss = -F.logsigmoid(score_diff).mean()

        if reg_lambda > 0:
            l2_penalty = model.l2_regularization(
                users,
                pos_items,
                neg_items,
            )
            loss = ranking_loss + reg_lambda * l2_penalty
        else:
            l2_penalty = torch.tensor(0.0, device=device)
            loss = ranking_loss

        loss.backward()
        optimizer.step()

        batch_size_actual = users.shape[0]

        total_loss += float(loss.detach().cpu()) * batch_size_actual
        total_ranking_loss += float(ranking_loss.detach().cpu()) * batch_size_actual
        total_l2_penalty += float(l2_penalty.detach().cpu()) * batch_size_actual
        total_examples += batch_size_actual

        if show_progress:
            iterator.set_postfix({
                "loss": total_loss / total_examples,
                "rank_loss": total_ranking_loss / total_examples,
            })

    return {
        "train_loss": total_loss / total_examples,
        "train_ranking_loss": total_ranking_loss / total_examples,
        "train_l2_penalty": total_l2_penalty / total_examples,
    }

### 7.2 Train model with early stopping

In [26]:
def train_bpr_experiment(
    config,
    train_dataset,
    val_dataset,
    test_dataset,
    train_pairs_idx,
    val_pairs_idx,
    n_users,
    n_items,
    device,
    seed,
    max_epochs=20,
    patience=3,
    eval_batch_size=65_536,
):
    set_all_seeds(seed)

    train_loader, val_loader, test_loader, loader_summary = make_pairwise_loaders(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        batch_size=config["batch_size"],
        seed=seed,
    )

    model = BPRMatrixFactorization(
        n_users=n_users,
        n_items=n_items,
        embedding_dim=config["embedding_dim"],
        init_std=config.get("init_std", 0.01),
        use_item_bias=config.get("use_item_bias", True),
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
    )

    best_metric = -np.inf
    best_epoch = None
    best_state_dict = None
    epochs_without_improvement = 0

    history_rows = []

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        epoch_start_time = time.time()

        train_metrics = train_one_epoch(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            device=device,
            reg_lambda=config["reg_lambda"],
            show_progress=config.get("show_progress", True),
        )

        val_metrics = evaluate_pairwise(
            model=model,
            pair_df=val_pairs_idx,
            device=device,
            split_name="validation",
            batch_size=eval_batch_size,
        )

        selection_metric = val_metrics["comparison_accuracy_macro_user"]

        epoch_time = time.time() - epoch_start_time

        row = {
            "experiment": config["experiment"],
            "epoch": epoch,
            "embedding_dim": config["embedding_dim"],
            "learning_rate": config["learning_rate"],
            "reg_lambda": config["reg_lambda"],
            "batch_size": config["batch_size"],
            "use_item_bias": config.get("use_item_bias", True),
            "epoch_time_sec": round(epoch_time, 2),
            **train_metrics,
            "val_comparison_accuracy_micro": val_metrics["comparison_accuracy_micro"],
            "val_comparison_accuracy_macro_user": val_metrics["comparison_accuracy_macro_user"],
            "val_roc_auc_symmetric": val_metrics["roc_auc_symmetric"],
            "val_log_loss_positive_pairs": val_metrics["log_loss_positive_pairs"],
            "val_brier_positive_pairs": val_metrics["brier_positive_pairs"],
            "val_mean_score_diff": val_metrics["mean_score_diff"],
            "val_median_score_diff": val_metrics["median_score_diff"],
        }

        history_rows.append(row)

        if selection_metric > best_metric:
            best_metric = selection_metric
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{config['experiment']} | "
            f"epoch {epoch:02d} | "
            f"train_loss={train_metrics['train_loss']:.4f} | "
            f"val_macro_acc={val_metrics['comparison_accuracy_macro_user']:.4f} | "
            f"val_auc={val_metrics['roc_auc_symmetric']:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        if epochs_without_improvement >= patience:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}, best val macro accuracy: {best_metric:.4f}"
            )
            break

    total_time = time.time() - start_time

    history = pd.DataFrame(history_rows)

    model.load_state_dict(best_state_dict)

    final_train_eval = evaluate_pairwise(
        model=model,
        pair_df=train_pairs_idx,
        device=device,
        split_name="train_best",
        batch_size=eval_batch_size,
    )

    final_val_eval = evaluate_pairwise(
        model=model,
        pair_df=val_pairs_idx,
        device=device,
        split_name="validation_best",
        batch_size=eval_batch_size,
    )

    final_summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_val_macro_accuracy": best_metric,
        "total_time_sec": round(total_time, 2),
        "embedding_dim": config["embedding_dim"],
        "learning_rate": config["learning_rate"],
        "reg_lambda": config["reg_lambda"],
        "batch_size": config["batch_size"],
        "use_item_bias": config.get("use_item_bias", True),
        "train_macro_accuracy_best": final_train_eval["comparison_accuracy_macro_user"],
        "train_micro_accuracy_best": final_train_eval["comparison_accuracy_micro"],
        "train_auc_best": final_train_eval["roc_auc_symmetric"],
        "validation_macro_accuracy_best": final_val_eval["comparison_accuracy_macro_user"],
        "validation_micro_accuracy_best": final_val_eval["comparison_accuracy_micro"],
        "validation_auc_best": final_val_eval["roc_auc_symmetric"],
        "validation_log_loss_best": final_val_eval["log_loss_positive_pairs"],
        "validation_brier_best": final_val_eval["brier_positive_pairs"],
    }

    return model, history, final_summary

# 8 Baseline training run

In [27]:
baseline_config = {
    "experiment": "baseline_dim32_lr1e-3_reg1e-5_bs16384_bias",
    "embedding_dim": 32,
    "learning_rate": 1e-3,
    "reg_lambda": 1e-5,
    "batch_size": 16_384,
    "init_std": 0.01,
    "use_item_bias": True,
    "show_progress": True,
}

baseline_model, baseline_history, baseline_summary = train_bpr_experiment(
    config=baseline_config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    train_pairs_idx=train_pairs_idx,
    val_pairs_idx=val_pairs_idx,
    n_users=n_users,
    n_items=n_items,
    device=device,
    seed=random_seed,
    max_epochs=20,
    patience=3,
)

display(baseline_history)
display(pd.DataFrame([baseline_summary]).round(4))

baseline_dim32_lr1e-3_reg1e-5_bs16384_bias | epoch 01 | train_loss=0.5256 | val_macro_acc=0.7273 | val_auc=0.8654 | time=180.3s


baseline_dim32_lr1e-3_reg1e-5_bs16384_bias | epoch 02 | train_loss=0.3482 | val_macro_acc=0.7477 | val_auc=0.8855 | time=174.1s


baseline_dim32_lr1e-3_reg1e-5_bs16384_bias | epoch 03 | train_loss=0.2715 | val_macro_acc=0.7514 | val_auc=0.8909 | time=172.5s


baseline_dim32_lr1e-3_reg1e-5_bs16384_bias | epoch 04 | train_loss=0.2146 | val_macro_acc=0.7502 | val_auc=0.8914 | time=171.7s


baseline_dim32_lr1e-3_reg1e-5_bs16384_bias | epoch 05 | train_loss=0.1743 | val_macro_acc=0.7455 | val_auc=0.8899 | time=172.3s


baseline_dim32_lr1e-3_reg1e-5_bs16384_bias | epoch 06 | train_loss=0.1464 | val_macro_acc=0.7432 | val_auc=0.8876 | time=172.5s
Early stopping at epoch 6. Best epoch: 3, best val macro accuracy: 0.7514


,experiment,epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,epoch_time_sec,train_loss,train_ranking_loss,train_l2_penalty,val_comparison_accuracy_micro,val_comparison_accuracy_macro_user,val_roc_auc_symmetric,val_log_loss_positive_pairs,val_brier_positive_pairs,val_mean_score_diff,val_median_score_diff
0,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,1,32,0.001,0.00001,16384,True,180.27,0.525611,0.525591,2.005169,0.776418,0.727295,0.865371,0.455626,0.150411,1.144811,0.925662
1,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,2,32,0.001,0.00001,16384,True,174.07,0.348230,0.348162,6.787151,0.796473,0.747688,0.885542,0.421390,0.137904,1.583915,1.283109
2,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,3,32,0.001,0.00001,16384,True,172.53,0.271461,0.271349,11.145278,0.802304,0.751448,0.890857,0.412738,0.134550,1.795427,1.477353
3,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,4,32,0.001,0.00001,16384,True,171.71,0.214622,0.214465,15.673382,0.802948,0.750246,0.891427,0.413854,0.134480,1.942993,1.626553
4,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,5,32,0.001,0.00001,16384,True,172.33,0.174260,0.174059,20.136996,0.801794,0.745490,0.889884,0.420843,0.136079,2.067327,1.757574
5,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,6,32,0.001,0.00001,16384,True,172.48,0.146417,0.146174,24.308652,0.800173,0.743211,0.887649,0.431117,0.138361,2.182092,1.882613


,experiment,best_epoch,best_val_macro_accuracy,total_time_sec,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,train_micro_accuracy_best,train_auc_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best
0,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,3,0.7514,1043.42,32,0.001,0.0,16384,True,0.9194,0.9111,0.9712,0.7514,0.8023,0.8909,0.4127,0.1345


### Baseline Diagnosis

In [28]:
baseline_diagnosis = pd.DataFrame([
    {
        "metric": "best_epoch",
        "value": baseline_summary["best_epoch"],
    },
    {
        "metric": "train_macro_accuracy_best",
        "value": baseline_summary["train_macro_accuracy_best"],
    },
    {
        "metric": "validation_macro_accuracy_best",
        "value": baseline_summary["validation_macro_accuracy_best"],
    },
    {
        "metric": "macro_generalization_gap",
        "value": round(
            baseline_summary["train_macro_accuracy_best"]
            - baseline_summary["validation_macro_accuracy_best"],
            4,
        ),
    },
    {
        "metric": "validation_micro_minus_macro",
        "value": round(
            baseline_summary["validation_micro_accuracy_best"]
            - baseline_summary["validation_macro_accuracy_best"],
            4,
        ),
    },
    {
        "metric": "validation_auc_best",
        "value": baseline_summary["validation_auc_best"],
    },
    {
        "metric": "validation_log_loss_best",
        "value": baseline_summary["validation_log_loss_best"],
    },
    {
        "metric": "validation_brier_best",
        "value": baseline_summary["validation_brier_best"],
    },
])

display(baseline_diagnosis)

,metric,value
0,best_epoch,3.000000
1,train_macro_accuracy_best,0.919380
2,validation_macro_accuracy_best,0.751448
3,macro_generalization_gap,0.167900
4,validation_micro_minus_macro,0.050900
5,validation_auc_best,0.890857
6,validation_log_loss_best,0.412738
7,validation_brier_best,0.134550


# 9 Regularization Sweep

### 9.1 Create experiment registry

In [29]:
experiment_results = {}

experiment_results[baseline_config["experiment"]] = {
    "model": baseline_model,
    "history": baseline_history.copy(),
    "summary": baseline_summary.copy(),
    "stage": "baseline",
}

### 9.2 Define regularization experiment plan

In [30]:
def format_float_for_name(value):
    if value == 0:
        return "0"
    return f"{value:.0e}".replace("-", "m")


reg_lambda_values = [
    0.0,
    1e-6,
    3e-5,
    1e-4,
]

reg_experiment_plan = []

for reg_lambda in reg_lambda_values:
    config = baseline_config.copy()
    config["reg_lambda"] = reg_lambda
    config["experiment"] = (
        f"reg_sweep_"
        f"dim{config['embedding_dim']}_"
        f"lr{format_float_for_name(config['learning_rate'])}_"
        f"reg{format_float_for_name(reg_lambda)}_"
        f"bs{config['batch_size']}_"
        f"{'bias' if config['use_item_bias'] else 'nobias'}"
    )

    reg_experiment_plan.append(config)

reg_experiment_plan_df = pd.DataFrame(reg_experiment_plan)
display(reg_experiment_plan_df)

,experiment,embedding_dim,learning_rate,reg_lambda,batch_size,init_std,use_item_bias,show_progress
0,reg_sweep_dim32_lr1em03_reg0_bs16384_bias,32,0.001,0.000000,16384,0.01,True,True
1,reg_sweep_dim32_lr1em03_reg1em06_bs16384_bias,32,0.001,0.000001,16384,0.01,True,True
2,reg_sweep_dim32_lr1em03_reg3em05_bs16384_bias,32,0.001,0.000030,16384,0.01,True,True
3,reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias,32,0.001,0.000100,16384,0.01,True,True


### 9.3 Run regularization experiments

In [31]:
reg_sweep_summaries = []
reg_sweep_histories = []

for config in reg_experiment_plan:
    print("=" * 100)
    print("Running experiment:", config["experiment"])
    print("=" * 100)

    model, history, summary = train_bpr_experiment(
        config=config,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        train_pairs_idx=train_pairs_idx,
        val_pairs_idx=val_pairs_idx,
        n_users=n_users,
        n_items=n_items,
        device=device,
        seed=random_seed,
        max_epochs=20,
        patience=3,
    )

    history = history.copy()
    history["stage"] = "regularization_sweep"

    summary = summary.copy()
    summary["stage"] = "regularization_sweep"

    experiment_results[config["experiment"]] = {
        "model": model,
        "history": history,
        "summary": summary,
        "stage": "regularization_sweep",
    }

    reg_sweep_histories.append(history)
    reg_sweep_summaries.append(summary)

Running experiment: reg_sweep_dim32_lr1em03_reg0_bs16384_bias


reg_sweep_dim32_lr1em03_reg0_bs16384_bias | epoch 01 | train_loss=0.5256 | val_macro_acc=0.7273 | val_auc=0.8654 | time=188.0s


reg_sweep_dim32_lr1em03_reg0_bs16384_bias | epoch 02 | train_loss=0.3481 | val_macro_acc=0.7477 | val_auc=0.8855 | time=170.3s


reg_sweep_dim32_lr1em03_reg0_bs16384_bias | epoch 03 | train_loss=0.2713 | val_macro_acc=0.7514 | val_auc=0.8909 | time=200.7s


reg_sweep_dim32_lr1em03_reg0_bs16384_bias | epoch 04 | train_loss=0.2144 | val_macro_acc=0.7503 | val_auc=0.8914 | time=176.9s


...

reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias | epoch 02 | train_loss=0.3491 | val_macro_acc=0.7477 | val_auc=0.8856 | time=176.8s


reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias | epoch 03 | train_loss=0.2729 | val_macro_acc=0.7516 | val_auc=0.8909 | time=179.5s


reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias | epoch 04 | train_loss=0.2166 | val_macro_acc=0.7503 | val_auc=0.8915 | time=178.5s


reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias | epoch 05 | train_loss=0.1767 | val_macro_acc=0.7454 | val_auc=0.8900 | time=179.7s


reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias | epoch 06 | train_loss=0.1492 | val_macro_acc=0.7433 | val_auc=0.8878 | time=180.1s
Early stopping at epoch 6. Best epoch: 3, best val macro accuracy: 0.7516


### 9.4 Compare regularization results

In [32]:
all_summaries_so_far = pd.DataFrame(
    [
        result["summary"]
        for result in experiment_results.values()
    ]
)

reg_comparison_cols = [
    "stage",
    "experiment",
    "best_epoch",
    "embedding_dim",
    "learning_rate",
    "reg_lambda",
    "batch_size",
    "use_item_bias",
    "train_macro_accuracy_best",
    "validation_macro_accuracy_best",
    "validation_micro_accuracy_best",
    "validation_auc_best",
    "validation_log_loss_best",
    "validation_brier_best",
    "total_time_sec",
]

reg_comparison = (
    all_summaries_so_far[reg_comparison_cols]
    .sort_values(
        ["validation_macro_accuracy_best", "validation_auc_best"],
        ascending=False,
    )
    .reset_index(drop=True)
)

reg_comparison["macro_generalization_gap"] = (
    reg_comparison["train_macro_accuracy_best"]
    - reg_comparison["validation_macro_accuracy_best"]
)

display(reg_comparison.round(6))

,stage,experiment,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,macro_generalization_gap
0,regularization_sweep,reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias,3,32,0.001,0.000100,16384,True,0.919164,0.751572,0.802365,0.890900,0.412634,0.134518,1072.08,0.167593
1,NaN,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,3,32,0.001,0.000010,16384,True,0.919380,0.751448,0.802304,0.890857,0.412738,0.134550,1043.42,0.167931
2,regularization_sweep,reg_sweep_dim32_lr1em03_reg0_bs16384_bias,3,32,0.001,0.000000,16384,True,0.919405,0.751435,0.802312,0.890852,0.412750,0.134553,1092.24,0.167970
3,regularization_sweep,reg_sweep_dim32_lr1em03_reg1em06_bs16384_bias,3,32,0.001,0.000001,16384,True,0.919403,0.751433,0.802306,0.890853,0.412749,0.134553,1107.07,0.167970
4,regularization_sweep,reg_sweep_dim32_lr1em03_reg3em05_bs16384_bias,3,32,0.001,0.000030,16384,True,0.919340,0.751432,0.802292,0.890867,0.412714,0.134542,1073.91,0.167908


### 9.5 Select best regularization value

In [33]:
best_reg_row = reg_comparison.iloc[0]

best_reg_experiment = best_reg_row["experiment"]
best_reg_lambda = float(best_reg_row["reg_lambda"])

print("Best experiment after regularization sweep:", best_reg_experiment)
print("Best reg_lambda:", best_reg_lambda)
print("Best validation macro accuracy:", best_reg_row["validation_macro_accuracy_best"])
print("Best validation AUC:", best_reg_row["validation_auc_best"])
print("Macro generalization gap:", best_reg_row["macro_generalization_gap"])

Best experiment after regularization sweep: reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias
Best reg_lambda: 0.0001
Best validation macro accuracy: 0.7515716701452893
Best validation AUC: 0.8908997266469815
Macro generalization gap: 0.1675927953192473


The regularization sweep does not identify a strongly dominant value.  
The best validation macro user-level pairwise accuracy is obtained with `reg_lambda = 1e-4`, but the differences between tested values are small.

We therefore use `reg_lambda = 1e-4` as the current incumbent value for the next staged experiments.  
This should not be interpreted as a globally optimal regularization strength, because we are not running an exhaustive grid over all hyperparameter combinations. Later focused joint experiments will re-check interactions between regularization, embedding dimension, learning rate, and batch size.

# 10 Embedding Dimension Sweep

### 10.1 Define embedding dimension experiment plan

In [34]:
# Clean baseline stage label in the global registry.
experiment_results[baseline_config["experiment"]]["summary"]["stage"] = "baseline"

embedding_dim_values = [16, 64]

embedding_experiment_plan = []

for embedding_dim in embedding_dim_values:
    config = baseline_config.copy()
    config["embedding_dim"] = embedding_dim
    config["reg_lambda"] = best_reg_lambda
    config["experiment"] = (
        f"dim_sweep_"
        f"dim{embedding_dim}_"
        f"lr{format_float_for_name(config['learning_rate'])}_"
        f"reg{format_float_for_name(config['reg_lambda'])}_"
        f"bs{config['batch_size']}_"
        f"{'bias' if config['use_item_bias'] else 'nobias'}"
    )

    embedding_experiment_plan.append(config)

embedding_experiment_plan_df = pd.DataFrame(embedding_experiment_plan)
display(embedding_experiment_plan_df)

,experiment,embedding_dim,learning_rate,reg_lambda,batch_size,init_std,use_item_bias,show_progress
0,dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias,16,0.001,0.0001,16384,0.01,True,True
1,dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias,64,0.001,0.0001,16384,0.01,True,True


### 10.2 Run embedding dimension experiments

In [35]:
embedding_sweep_summaries = []
embedding_sweep_histories = []

for config in embedding_experiment_plan:
    print("=" * 100)
    print("Running experiment:", config["experiment"])
    print("=" * 100)

    model, history, summary = train_bpr_experiment(
        config=config,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        train_pairs_idx=train_pairs_idx,
        val_pairs_idx=val_pairs_idx,
        n_users=n_users,
        n_items=n_items,
        device=device,
        seed=random_seed,
        max_epochs=20,
        patience=3,
    )

    history = history.copy()
    history["stage"] = "embedding_dim_sweep"

    summary = summary.copy()
    summary["stage"] = "embedding_dim_sweep"

    experiment_results[config["experiment"]] = {
        "model": model,
        "history": history,
        "summary": summary,
        "stage": "embedding_dim_sweep",
    }

    embedding_sweep_histories.append(history)
    embedding_sweep_summaries.append(summary)

Running experiment: dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias


dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias | epoch 01 | train_loss=0.5561 | val_macro_acc=0.7213 | val_auc=0.8577 | time=192.8s


dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias | epoch 02 | train_loss=0.3895 | val_macro_acc=0.7396 | val_auc=0.8783 | time=213.3s


dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias | epoch 03 | train_loss=0.3357 | val_macro_acc=0.7443 | val_auc=0.8850 | time=213.4s


dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias | epoch 04 | train_loss=0.2956 | val_macro_acc=0.7457 | val_auc=0.8874 | time=175.0s


...

dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias | epoch 01 | train_loss=0.4918 | val_macro_acc=0.7375 | val_auc=0.8761 | time=194.3s


dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias | epoch 02 | train_loss=0.2827 | val_macro_acc=0.7523 | val_auc=0.8923 | time=193.0s


dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias | epoch 03 | train_loss=0.1875 | val_macro_acc=0.7515 | val_auc=0.8943 | time=194.4s


dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias | epoch 04 | train_loss=0.1308 | val_macro_acc=0.7465 | val_auc=0.8922 | time=193.4s


dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias | epoch 05 | train_loss=0.0975 | val_macro_acc=0.7414 | val_auc=0.8889 | time=193.8s
Early stopping at epoch 5. Best epoch: 2, best val macro accuracy: 0.7523


### 10.3 Compare embedding dimension results

In [36]:
all_summaries_so_far = pd.DataFrame(
    [
        result["summary"]
        for result in experiment_results.values()
    ]
)

embedding_comparison_cols = [
    "stage",
    "experiment",
    "best_epoch",
    "embedding_dim",
    "learning_rate",
    "reg_lambda",
    "batch_size",
    "use_item_bias",
    "train_macro_accuracy_best",
    "validation_macro_accuracy_best",
    "validation_micro_accuracy_best",
    "validation_auc_best",
    "validation_log_loss_best",
    "validation_brier_best",
    "total_time_sec",
]

embedding_comparison = (
    all_summaries_so_far[embedding_comparison_cols]
    .sort_values(
        ["validation_macro_accuracy_best", "validation_auc_best"],
        ascending=False,
    )
    .reset_index(drop=True)
)

embedding_comparison["macro_generalization_gap"] = (
    embedding_comparison["train_macro_accuracy_best"]
    - embedding_comparison["validation_macro_accuracy_best"]
)

display(embedding_comparison.round(6))

,stage,experiment,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,macro_generalization_gap
0,embedding_dim_sweep,dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias,2,64,0.001,0.000100,16384,True,0.931816,0.752302,0.803783,0.892300,0.409997,0.133594,968.92,0.179515
1,regularization_sweep,reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias,3,32,0.001,0.000100,16384,True,0.919164,0.751572,0.802365,0.890900,0.412634,0.134518,1072.08,0.167593
2,baseline,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,3,32,0.001,0.000010,16384,True,0.919380,0.751448,0.802304,0.890857,0.412738,0.134550,1043.42,0.167931
3,regularization_sweep,reg_sweep_dim32_lr1em03_reg0_bs16384_bias,3,32,0.001,0.000000,16384,True,0.919405,0.751435,0.802312,0.890852,0.412750,0.134553,1092.24,0.167970
4,regularization_sweep,reg_sweep_dim32_lr1em03_reg1em06_bs16384_bias,3,32,0.001,0.000001,16384,True,0.919403,0.751433,0.802306,0.890853,0.412749,0.134553,1107.07,0.167970
5,regularization_sweep,reg_sweep_dim32_lr1em03_reg3em05_bs16384_bias,3,32,0.001,0.000030,16384,True,0.919340,0.751432,0.802292,0.890867,0.412714,0.134542,1073.91,0.167908
6,embedding_dim_sweep,dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias,4,16,0.001,0.000100,16384,True,0.894891,0.745718,0.798058,0.887394,0.418695,0.136816,1336.47,0.149173


### 10.4 Select best embedding dimension

In [37]:
best_embedding_row = embedding_comparison.iloc[0]

best_embedding_experiment = best_embedding_row["experiment"]
best_embedding_dim = int(best_embedding_row["embedding_dim"])

print("Best experiment after embedding sweep:", best_embedding_experiment)
print("Best embedding_dim:", best_embedding_dim)
print("Best reg_lambda:", best_embedding_row["reg_lambda"])
print("Best validation macro accuracy:", best_embedding_row["validation_macro_accuracy_best"])
print("Best validation AUC:", best_embedding_row["validation_auc_best"])
print("Macro generalization gap:", best_embedding_row["macro_generalization_gap"])

Best experiment after embedding sweep: dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias
Best embedding_dim: 64
Best reg_lambda: 0.0001
Best validation macro accuracy: 0.7523015897672283
Best validation AUC: 0.8923002126189532
Macro generalization gap: 0.1795148517116617


# 11 Learning Rate Sweep

### 11.1 Define learning rate experiment plan

In [38]:
learning_rate_values = [
    3e-4,
    3e-3,
]

learning_rate_experiment_plan = []

for learning_rate in learning_rate_values:
    config = baseline_config.copy()

    config["embedding_dim"] = best_embedding_dim
    config["reg_lambda"] = best_reg_lambda
    config["learning_rate"] = learning_rate

    config["experiment"] = (
        f"lr_sweep_"
        f"dim{config['embedding_dim']}_"
        f"lr{format_float_for_name(config['learning_rate'])}_"
        f"reg{format_float_for_name(config['reg_lambda'])}_"
        f"bs{config['batch_size']}_"
        f"{'bias' if config['use_item_bias'] else 'nobias'}"
    )

    learning_rate_experiment_plan.append(config)

learning_rate_experiment_plan_df = pd.DataFrame(learning_rate_experiment_plan)
display(learning_rate_experiment_plan_df)

,experiment,embedding_dim,learning_rate,reg_lambda,batch_size,init_std,use_item_bias,show_progress
0,lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias,64,0.0003,0.0001,16384,0.01,True,True
1,lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias,64,0.0030,0.0001,16384,0.01,True,True


### 11.2 Run learning rate experiments

In [39]:
learning_rate_sweep_summaries = []
learning_rate_sweep_histories = []

for config in learning_rate_experiment_plan:
    print("=" * 100)
    print("Running experiment:", config["experiment"])
    print("=" * 100)

    model, history, summary = train_bpr_experiment(
        config=config,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        train_pairs_idx=train_pairs_idx,
        val_pairs_idx=val_pairs_idx,
        n_users=n_users,
        n_items=n_items,
        device=device,
        seed=random_seed,
        max_epochs=20,
        patience=3,
    )

    history = history.copy()
    history["stage"] = "learning_rate_sweep"

    summary = summary.copy()
    summary["stage"] = "learning_rate_sweep"

    experiment_results[config["experiment"]] = {
        "model": model,
        "history": history,
        "summary": summary,
        "stage": "learning_rate_sweep",
    }

    learning_rate_sweep_histories.append(history)
    learning_rate_sweep_summaries.append(summary)

Running experiment: lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias


lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias | epoch 01 | train_loss=0.6347 | val_macro_acc=0.6990 | val_auc=0.8370 | time=214.1s


lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias | epoch 02 | train_loss=0.4559 | val_macro_acc=0.7199 | val_auc=0.8603 | time=198.5s


lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias | epoch 03 | train_loss=0.3890 | val_macro_acc=0.7350 | val_auc=0.8741 | time=205.1s


lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias | epoch 04 | train_loss=0.3462 | val_macro_acc=0.7452 | val_auc=0.8826 | time=197.1s


...

lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias | epoch 13 | train_loss=0.1295 | val_macro_acc=0.7526 | val_auc=0.8955 | time=195.7s
Early stopping at epoch 13. Best epoch: 10, best val macro accuracy: 0.7567
Running experiment: lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias


lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias | epoch 01 | train_loss=0.3313 | val_macro_acc=0.7450 | val_auc=0.8905 | time=193.6s


lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias | epoch 02 | train_loss=0.1018 | val_macro_acc=0.7342 | val_auc=0.8812 | time=192.0s


lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias | epoch 03 | train_loss=0.0562 | val_macro_acc=0.7262 | val_auc=0.8733 | time=191.6s


lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias | epoch 04 | train_loss=0.0394 | val_macro_acc=0.7217 | val_auc=0.8678 | time=166.3s
Early stopping at epoch 4. Best epoch: 1, best val macro accuracy: 0.7450


### 11.3 Compare learning rate results

In [40]:
all_summaries_so_far = pd.DataFrame(
    [
        result["summary"]
        for result in experiment_results.values()
    ]
)

learning_rate_comparison_cols = [
    "stage",
    "experiment",
    "best_epoch",
    "embedding_dim",
    "learning_rate",
    "reg_lambda",
    "batch_size",
    "use_item_bias",
    "train_macro_accuracy_best",
    "validation_macro_accuracy_best",
    "validation_micro_accuracy_best",
    "validation_auc_best",
    "validation_log_loss_best",
    "validation_brier_best",
    "total_time_sec",
]

learning_rate_comparison = (
    all_summaries_so_far[learning_rate_comparison_cols]
    .sort_values(
        ["validation_macro_accuracy_best", "validation_auc_best"],
        ascending=False,
    )
    .reset_index(drop=True)
)

learning_rate_comparison["macro_generalization_gap"] = (
    learning_rate_comparison["train_macro_accuracy_best"]
    - learning_rate_comparison["validation_macro_accuracy_best"]
)

display(learning_rate_comparison.round(6))

,stage,experiment,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,macro_generalization_gap
0,learning_rate_sweep,lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias,10,64,0.0003,0.000100,16384,True,0.957874,0.756726,0.808635,0.896333,0.403846,0.131054,2574.66,0.201148
1,embedding_dim_sweep,dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias,2,64,0.0010,0.000100,16384,True,0.931816,0.752302,0.803783,0.892300,0.409997,0.133594,968.92,0.179515
2,regularization_sweep,reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias,3,32,0.0010,0.000100,16384,True,0.919164,0.751572,0.802365,0.890900,0.412634,0.134518,1072.08,0.167593
3,baseline,baseline_dim32_lr1e-3_reg1e-5_bs16384_bias,3,32,0.0010,0.000010,16384,True,0.919380,0.751448,0.802304,0.890857,0.412738,0.134550,1043.42,0.167931
4,regularization_sweep,reg_sweep_dim32_lr1em03_reg0_bs16384_bias,3,32,0.0010,0.000000,16384,True,0.919405,0.751435,0.802312,0.890852,0.412750,0.134553,1092.24,0.167970
5,regularization_sweep,reg_sweep_dim32_lr1em03_reg1em06_bs16384_bias,3,32,0.0010,0.000001,16384,True,0.919403,0.751433,0.802306,0.890853,0.412749,0.134553,1107.07,0.167970
6,regularization_sweep,reg_sweep_dim32_lr1em03_reg3em05_bs16384_bias,3,32,0.0010,0.000030,16384,True,0.919340,0.751432,0.802292,0.890867,0.412714,0.134542,1073.91,0.167908
7,embedding_dim_sweep,dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias,4,16,0.0010,0.000100,16384,True,0.894891,0.745718,0.798058,0.887394,0.418695,0.136816,1336.47,0.149173
8,learning_rate_sweep,lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias,1,64,0.0030,0.000100,16384,True,0.967091,0.745033,0.802624,0.890479,0.414888,0.135028,743.51,0.222058


### 11.4 Select best learning rate

In [41]:
best_lr_row = learning_rate_comparison.iloc[0]

best_lr_experiment = best_lr_row["experiment"]
best_learning_rate = float(best_lr_row["learning_rate"])

print("Best experiment after learning rate sweep:", best_lr_experiment)
print("Best embedding_dim:", int(best_lr_row["embedding_dim"]))
print("Best learning_rate:", best_learning_rate)
print("Best reg_lambda:", best_lr_row["reg_lambda"])
print("Best validation macro accuracy:", best_lr_row["validation_macro_accuracy_best"])
print("Best validation AUC:", best_lr_row["validation_auc_best"])
print("Macro generalization gap:", best_lr_row["macro_generalization_gap"])

Best experiment after learning rate sweep: lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias
Best embedding_dim: 64
Best learning_rate: 0.0003
Best reg_lambda: 0.0001
Best validation macro accuracy: 0.756726201775643
Best validation AUC: 0.896332583115557
Macro generalization gap: 0.2011475547320838


# 12 Batch Size Sweep

We now tune the mini-batch size while keeping the best settings selected so far:

- embedding dimension from the embedding sweep
- learning rate from the learning-rate sweep
- regularization from the regularization sweep
- item bias setting unchanged

Batch size can affect both runtime and optimization dynamics. Smaller batches produce more optimizer updates per epoch and noisier gradients. Larger batches produce fewer updates per epoch and may train faster, but they can change validation quality.

Therefore, batch size is selected using validation macro accuracy as the primary metric, with runtime used as a tie-breaker when validation performance is effectively equal.

### 12.1 Define batch size experiment plan

In [42]:
# Current best configuration after previous sweeps.
# These variables should already exist after blocks 9-11:
# best_embedding_dim
# best_learning_rate
# best_reg_lambda

batch_size_values = [
    8_192,
    16_384,
    32_768,
    65_536,
]

batch_experiment_plan = []

for batch_size_value in batch_size_values:
    config = baseline_config.copy()

    config["embedding_dim"] = int(best_embedding_dim)
    config["learning_rate"] = float(best_learning_rate)
    config["reg_lambda"] = float(best_reg_lambda)
    config["batch_size"] = int(batch_size_value)

    config["experiment"] = (
        f"bs_sweep_"
        f"dim{config['embedding_dim']}_"
        f"lr{format_float_for_name(config['learning_rate'])}_"
        f"reg{format_float_for_name(config['reg_lambda'])}_"
        f"bs{config['batch_size']}_"
        f"{'bias' if config['use_item_bias'] else 'nobias'}"
    )

    batch_experiment_plan.append(config)

batch_experiment_plan_df = pd.DataFrame(batch_experiment_plan)
display(batch_experiment_plan_df)

,experiment,embedding_dim,learning_rate,reg_lambda,batch_size,init_std,use_item_bias,show_progress
0,bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias,64,0.0003,0.0001,8192,0.01,True,True
1,bs_sweep_dim64_lr3em04_reg1em04_bs16384_bias,64,0.0003,0.0001,16384,0.01,True,True
2,bs_sweep_dim64_lr3em04_reg1em04_bs32768_bias,64,0.0003,0.0001,32768,0.01,True,True
3,bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias,64,0.0003,0.0001,65536,0.01,True,True


### 12.2 Helper: reuse already trained matching experiments

In [43]:
def configs_match_summary(summary, config, atol=1e-12):
    return (
        int(summary["embedding_dim"]) == int(config["embedding_dim"])
        and int(summary["batch_size"]) == int(config["batch_size"])
        and bool(summary["use_item_bias"]) == bool(config["use_item_bias"])
        and abs(float(summary["learning_rate"]) - float(config["learning_rate"])) <= atol
        and abs(float(summary["reg_lambda"]) - float(config["reg_lambda"])) <= atol
    )


def find_existing_matching_experiment(experiment_results, config):
    matches = []

    for experiment_name, result in experiment_results.items():
        summary = result["summary"]

        if configs_match_summary(summary, config):
            matches.append((experiment_name, result))

    if not matches:
        return None, None

    # If several previous experiments have the same hyperparameters,
    # keep the one with the best validation macro accuracy.
    matches = sorted(
        matches,
        key=lambda x: x[1]["summary"]["validation_macro_accuracy_best"],
        reverse=True,
    )

    return matches[0]

### 12.3 Run batch size experiments

In [44]:
batch_sweep_summaries = []
batch_sweep_histories = []
batch_sweep_failures = []

for config in batch_experiment_plan:
    print("=" * 100)
    print("Batch size experiment:", config["experiment"])
    print("=" * 100)

    existing_name, existing_result = find_existing_matching_experiment(
        experiment_results,
        config,
    )

    if existing_result is not None:
        print(f"Reusing existing trained experiment: {existing_name}")

        model = existing_result["model"]

        history = existing_result["history"].copy()
        history["stage"] = "batch_size_sweep_reused"
        history["reused_from_experiment"] = existing_name

        summary = existing_result["summary"].copy()
        summary["experiment"] = config["experiment"]
        summary["stage"] = "batch_size_sweep_reused"
        summary["reused_from_experiment"] = existing_name

        experiment_results[config["experiment"]] = {
            "model": model,
            "history": history,
            "summary": summary,
            "stage": "batch_size_sweep_reused",
            "reused_from_experiment": existing_name,
        }

    else:
        try:
            model, history, summary = train_bpr_experiment(
                config=config,
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                test_dataset=test_dataset,
                train_pairs_idx=train_pairs_idx,
                val_pairs_idx=val_pairs_idx,
                n_users=n_users,
                n_items=n_items,
                device=device,
                seed=random_seed,
                max_epochs=20,
                patience=3,
            )

            history = history.copy()
            history["stage"] = "batch_size_sweep"

            summary = summary.copy()
            summary["stage"] = "batch_size_sweep"

            experiment_results[config["experiment"]] = {
                "model": model,
                "history": history,
                "summary": summary,
                "stage": "batch_size_sweep",
            }

        except RuntimeError as error:
            error_message = str(error)

            print("Experiment failed:", config["experiment"])
            print(error_message)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            failure_row = config.copy()
            failure_row["stage"] = "batch_size_sweep"
            failure_row["error"] = error_message

            batch_sweep_failures.append(failure_row)
            continue

    batch_sweep_histories.append(history)
    batch_sweep_summaries.append(summary)

Batch size experiment: bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias


bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias | epoch 01 | train_loss=0.5692 | val_macro_acc=0.7129 | val_auc=0.8530 | time=213.2s


bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias | epoch 02 | train_loss=0.3932 | val_macro_acc=0.7380 | val_auc=0.8757 | time=208.9s


bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias | epoch 03 | train_loss=0.3306 | val_macro_acc=0.7509 | val_auc=0.8862 | time=209.5s


bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias | epoch 04 | train_loss=0.2822 | val_macro_acc=0.7548 | val_auc=0.8915 | time=208.2s


...

bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias | epoch 16 | train_loss=0.2686 | val_macro_acc=0.7521 | val_auc=0.8906 | time=166.5s


bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias | epoch 17 | train_loss=0.2559 | val_macro_acc=0.7527 | val_auc=0.8917 | time=165.8s


bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias | epoch 18 | train_loss=0.2437 | val_macro_acc=0.7529 | val_auc=0.8926 | time=170.5s


bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias | epoch 19 | train_loss=0.2321 | val_macro_acc=0.7541 | val_auc=0.8933 | time=212.2s


bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias | epoch 20 | train_loss=0.2209 | val_macro_acc=0.7558 | val_auc=0.8939 | time=198.5s


### 12.4 Compare batch size results

In [45]:
all_summaries_so_far = pd.DataFrame(
    [
        result["summary"]
        for result in experiment_results.values()
    ]
)

batch_comparison_cols = [
    "stage",
    "experiment",
    "best_epoch",
    "embedding_dim",
    "learning_rate",
    "reg_lambda",
    "batch_size",
    "use_item_bias",
    "train_macro_accuracy_best",
    "validation_macro_accuracy_best",
    "validation_micro_accuracy_best",
    "validation_auc_best",
    "validation_log_loss_best",
    "validation_brier_best",
    "total_time_sec",
]

batch_comparison = (
    all_summaries_so_far[
        all_summaries_so_far["experiment"].isin(
            [config["experiment"] for config in batch_experiment_plan]
        )
    ][batch_comparison_cols]
    .copy()
)

batch_comparison["n_train_batches_per_epoch"] = np.ceil(
    len(train_dataset) / batch_comparison["batch_size"]
).astype(int)

batch_comparison["optimizer_updates_to_best_epoch"] = (
    batch_comparison["best_epoch"]
    * batch_comparison["n_train_batches_per_epoch"]
).astype(int)

batch_comparison["macro_generalization_gap"] = (
    batch_comparison["train_macro_accuracy_best"]
    - batch_comparison["validation_macro_accuracy_best"]
)

batch_comparison = (
    batch_comparison
    .sort_values(
        [
            "validation_macro_accuracy_best",
            "validation_auc_best",
            "total_time_sec",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

display(batch_comparison.round(6))

,stage,experiment,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,n_train_batches_per_epoch,optimizer_updates_to_best_epoch,macro_generalization_gap
0,batch_size_sweep,bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias,6,64,0.0003,0.0001,8192,True,0.948321,0.757608,0.808433,0.896024,0.404075,0.131186,1887.21,695,4170,0.190713
1,batch_size_sweep_reused,bs_sweep_dim64_lr3em04_reg1em04_bs16384_bias,10,64,0.0003,0.0001,16384,True,0.957874,0.756726,0.808635,0.896333,0.403846,0.131054,2574.66,348,3480,0.201148
2,batch_size_sweep,bs_sweep_dim64_lr3em04_reg1em04_bs32768_bias,14,64,0.0003,0.0001,32768,True,0.949681,0.756656,0.807309,0.895551,0.404601,0.131510,3403.24,174,2436,0.193025
3,batch_size_sweep,bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias,20,64,0.0003,0.0001,65536,True,0.937226,0.755769,0.805655,0.893888,0.407325,0.132552,3635.83,87,1740,0.181457


### 12.5 Select best batch size

In [46]:
# Primary rule:
# choose the fastest batch size within this tolerance of the best validation macro accuracy.
#
# 0.001 means: if a batch size is within 0.1 percentage point of the best macro accuracy,
# we treat it as practically tied and prefer the faster configuration.

batch_macro_tolerance = 0.001

best_batch_macro = batch_comparison["validation_macro_accuracy_best"].max()

batch_candidates = batch_comparison[
    batch_comparison["validation_macro_accuracy_best"]
    >= best_batch_macro - batch_macro_tolerance
].copy()

batch_candidates = (
    batch_candidates
    .sort_values(
        [
            "total_time_sec",
            "validation_auc_best",
            "validation_macro_accuracy_best",
        ],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

best_batch_row = batch_candidates.iloc[0]

best_batch_experiment = best_batch_row["experiment"]
best_batch_size = int(best_batch_row["batch_size"])

print("Best experiment after batch size sweep:", best_batch_experiment)
print("Best batch_size:", best_batch_size)
print("Best validation macro accuracy:", best_batch_row["validation_macro_accuracy_best"])
print("Best validation AUC:", best_batch_row["validation_auc_best"])
print("Total time sec:", best_batch_row["total_time_sec"])
print("Best epoch:", int(best_batch_row["best_epoch"]))
print("Optimizer updates to best epoch:", int(best_batch_row["optimizer_updates_to_best_epoch"]))
print("Macro generalization gap:", best_batch_row["macro_generalization_gap"])

display(batch_candidates.round(6))

Best experiment after batch size sweep: bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias
Best batch_size: 8192
Best validation macro accuracy: 0.7576079589130686
Best validation AUC: 0.8960241608384103
Total time sec: 1887.21
Best epoch: 6
Optimizer updates to best epoch: 4170
Macro generalization gap: 0.19071335542136858


,stage,experiment,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,n_train_batches_per_epoch,optimizer_updates_to_best_epoch,macro_generalization_gap
0,batch_size_sweep,bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias,6,64,0.0003,0.0001,8192,True,0.948321,0.757608,0.808433,0.896024,0.404075,0.131186,1887.21,695,4170,0.190713
1,batch_size_sweep_reused,bs_sweep_dim64_lr3em04_reg1em04_bs16384_bias,10,64,0.0003,0.0001,16384,True,0.957874,0.756726,0.808635,0.896333,0.403846,0.131054,2574.66,348,3480,0.201148
2,batch_size_sweep,bs_sweep_dim64_lr3em04_reg1em04_bs32768_bias,14,64,0.0003,0.0001,32768,True,0.949681,0.756656,0.807309,0.895551,0.404601,0.131510,3403.24,174,2436,0.193025


### 12.6 Update current best config

In [47]:
current_best_config = baseline_config.copy()

current_best_config["embedding_dim"] = int(best_embedding_dim)
current_best_config["learning_rate"] = float(best_learning_rate)
current_best_config["reg_lambda"] = float(best_reg_lambda)
current_best_config["batch_size"] = int(best_batch_size)

current_best_config["experiment"] = (
    f"current_best_after_batch_sweep_"
    f"dim{current_best_config['embedding_dim']}_"
    f"lr{format_float_for_name(current_best_config['learning_rate'])}_"
    f"reg{format_float_for_name(current_best_config['reg_lambda'])}_"
    f"bs{current_best_config['batch_size']}_"
    f"{'bias' if current_best_config['use_item_bias'] else 'nobias'}"
)

current_best_summary = pd.DataFrame([
    {
        "parameter": "embedding_dim",
        "value": current_best_config["embedding_dim"],
    },
    {
        "parameter": "learning_rate",
        "value": current_best_config["learning_rate"],
    },
    {
        "parameter": "reg_lambda",
        "value": current_best_config["reg_lambda"],
    },
    {
        "parameter": "batch_size",
        "value": current_best_config["batch_size"],
    },
    {
        "parameter": "use_item_bias",
        "value": current_best_config["use_item_bias"],
    },
    {
        "parameter": "selection_rule",
        "value": f"fastest within {batch_macro_tolerance} validation macro accuracy of best",
    },
])

display(current_best_summary)

,parameter,value
0,embedding_dim,64
1,learning_rate,0.0003
2,reg_lambda,0.0001
3,batch_size,8192
4,use_item_bias,True
5,selection_rule,fastest within 0.001 validation macro accuracy...


# 13 Pair Sampling / Negative Sampling Ablation
### 13.1 Define pair generation helpers

In [48]:
from collections import defaultdict
from itertools import product
import math
import random


def sample_user_pairs_by_strategy(
    user_pairs,
    max_pairs,
    seed,
    sampling_strategy,
):
    """
    Samples user-level pair list.

    sampling_strategy:
    - "random_cap": simple random cap
    - "stratified_cap": proportional stratified cap by rating_diff
    """

    if max_pairs is None or len(user_pairs) <= max_pairs:
        return user_pairs

    rng = random.Random(seed)

    if sampling_strategy == "random_cap":
        return rng.sample(user_pairs, max_pairs)

    if sampling_strategy != "stratified_cap":
        raise ValueError(f"Unknown sampling_strategy: {sampling_strategy}")

    buckets = defaultdict(list)

    for row in user_pairs:
        buckets[int(row["rating_diff"])].append(row)

    total_pairs = len(user_pairs)

    quota_rows = []

    for rating_diff, rows in buckets.items():
        raw_quota = len(rows) * max_pairs / total_pairs
        base_quota = int(math.floor(raw_quota))

        # Keep at least one example from every non-empty rating-diff bucket.
        base_quota = max(1, base_quota)

        quota_rows.append({
            "rating_diff": rating_diff,
            "rows": rows,
            "raw_quota": raw_quota,
            "quota": min(base_quota, len(rows)),
            "fraction": raw_quota - math.floor(raw_quota),
        })

    # If quota sum is too large, remove from smallest fractional remainders.
    while sum(x["quota"] for x in quota_rows) > max_pairs:
        candidates = [x for x in quota_rows if x["quota"] > 1]

        if not candidates:
            break

        to_reduce = min(
            candidates,
            key=lambda x: (x["quota"], x["fraction"]),
        )

        to_reduce["quota"] -= 1

    # If quota sum is too small, add to largest fractional remainders.
    remaining = max_pairs - sum(x["quota"] for x in quota_rows)

    quota_rows = sorted(
        quota_rows,
        key=lambda x: x["fraction"],
        reverse=True,
    )

    while remaining > 0:
        progressed = False

        for x in quota_rows:
            if remaining == 0:
                break

            if x["quota"] < len(x["rows"]):
                x["quota"] += 1
                remaining -= 1
                progressed = True

        if not progressed:
            break

    sampled = []

    for x in quota_rows:
        rows = x["rows"]
        quota = x["quota"]

        if quota >= len(rows):
            sampled.extend(rows)
        else:
            sampled.extend(rng.sample(rows, quota))

    if len(sampled) > max_pairs:
        sampled = rng.sample(sampled, max_pairs)

    return sampled


def generate_pairwise_pairs_from_interactions(
    df,
    pair_min_diff,
    min_pairs_per_user,
    max_pairs_per_user,
    sampling_strategy,
    seed,
):
    """
    Generates pairwise preference data from explicit ratings.

    Positive item = higher-rated observed item.
    Negative item = lower-rated observed item.

    This is an observed-negative scheme, not random-unobserved negative sampling.
    """

    required_cols = ["user_id", "movie_id", "rating"]

    missing_cols = [
        col
        for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(f"Missing required interaction columns: {missing_cols}")

    all_rows = []

    user_summaries = []

    for user_id, user_df in df.groupby("user_id"):
        rating_buckets = {}

        for rating, bucket in user_df.groupby("rating"):
            rating_buckets[int(rating)] = (
                bucket["movie_id"]
                .astype(int)
                .tolist()
            )

        user_pairs = []

        for high_rating in sorted(rating_buckets.keys(), reverse=True):
            high_items = rating_buckets[high_rating]

            for low_rating in sorted(rating_buckets.keys(), reverse=True):
                rating_diff = int(high_rating - low_rating)

                if rating_diff < pair_min_diff:
                    continue

                low_items = rating_buckets[low_rating]

                for pos_item, neg_item in product(high_items, low_items):
                    if pos_item == neg_item:
                        continue

                    user_pairs.append({
                        "user_id": int(user_id),
                        "pos_item": int(pos_item),
                        "neg_item": int(neg_item),
                        "rating_pos": int(high_rating),
                        "rating_neg": int(low_rating),
                        "rating_diff": int(rating_diff),
                    })

        raw_pair_count = len(user_pairs)

        if min_pairs_per_user is not None and raw_pair_count < min_pairs_per_user:
            user_summaries.append({
                "user_id": int(user_id),
                "raw_pairs": raw_pair_count,
                "kept_pairs": 0,
                "removed_by_min_pairs": True,
                "affected_by_cap": False,
            })

            continue

        affected_by_cap = (
            max_pairs_per_user is not None
            and raw_pair_count > max_pairs_per_user
        )

        if affected_by_cap:
            user_seed = seed + int(user_id) * 1009 + int(pair_min_diff) * 9173

            user_pairs = sample_user_pairs_by_strategy(
                user_pairs=user_pairs,
                max_pairs=max_pairs_per_user,
                seed=user_seed,
                sampling_strategy=sampling_strategy,
            )

        user_summaries.append({
            "user_id": int(user_id),
            "raw_pairs": raw_pair_count,
            "kept_pairs": len(user_pairs),
            "removed_by_min_pairs": False,
            "affected_by_cap": bool(affected_by_cap),
        })

        all_rows.extend(user_pairs)

    pair_df = pd.DataFrame(all_rows)

    if len(pair_df) > 0:
        pair_df = (
            pair_df
            .drop_duplicates(
                subset=["user_id", "pos_item", "neg_item"]
            )
            .sort_values(
                ["user_id", "rating_diff", "pos_item", "neg_item"],
                ascending=[True, False, True, True],
            )
            .reset_index(drop=True)
        )

    user_summary_df = pd.DataFrame(user_summaries)

    return pair_df, user_summary_df


def summarize_pair_sampling_variant(pair_df, user_summary_df, variant_name):
    if len(pair_df) == 0:
        return {
            "pair_sampling_scheme": variant_name,
            "pairs": 0,
            "users_with_pairs": 0,
            "items_in_pairs": 0,
            "avg_pairs_per_user": 0.0,
            "median_pairs_per_user": 0.0,
            "min_pairs_per_user": 0,
            "max_pairs_per_user": 0,
            "users_removed_by_min_pairs": int(
                user_summary_df["removed_by_min_pairs"].sum()
            ),
            "users_affected_by_cap": int(
                user_summary_df["affected_by_cap"].sum()
            ),
        }

    user_pair_counts = pair_df.groupby("user_id").size()

    return {
        "pair_sampling_scheme": variant_name,
        "pairs": len(pair_df),
        "users_with_pairs": pair_df["user_id"].nunique(),
        "items_in_pairs": len(
            set(pair_df["pos_item"].unique())
            | set(pair_df["neg_item"].unique())
        ),
        "avg_pairs_per_user": round(user_pair_counts.mean(), 2),
        "median_pairs_per_user": round(user_pair_counts.median(), 2),
        "min_pairs_per_user": int(user_pair_counts.min()),
        "max_pairs_per_user": int(user_pair_counts.max()),
        "users_removed_by_min_pairs": int(
            user_summary_df["removed_by_min_pairs"].sum()
        ),
        "users_affected_by_cap": int(
            user_summary_df["affected_by_cap"].sum()
        ),
    }

### 13.2 Define pair-sampling ablation plan

In [49]:
# Recover Step 1 pair-construction settings.
train_min_pairs_per_user = int(
    step1_config.get(
        "train_min_pairs_per_user",
        step1_config.get("min_pairs_per_user", 10),
    )
)

train_max_pairs_per_user = int(
    step1_config.get(
        "train_max_pairs_per_user",
        step1_config.get("max_pairs_per_user", 2000),
    )
)

pair_ablation_seed = random_seed + 13

pair_sampling_ablation_plan = [
    {
        "pair_sampling_scheme": "current_step1_gap2_stratified_cap",
        "source": "existing_step1_pairs",
        "pair_min_diff": 2,
        "min_pairs_per_user": train_min_pairs_per_user,
        "max_pairs_per_user": train_max_pairs_per_user,
        "sampling_strategy": "stratified_cap",
        "description": "Original Step 1 policy: observed lower-rated negatives, rating_diff >= 2, stratified user-level cap.",
    },
    {
        "pair_sampling_scheme": "gap2_random_cap",
        "source": "generate_from_train_df",
        "pair_min_diff": 2,
        "min_pairs_per_user": train_min_pairs_per_user,
        "max_pairs_per_user": train_max_pairs_per_user,
        "sampling_strategy": "random_cap",
        "description": "Same threshold and cap as Step 1, but random user-level cap instead of stratified cap.",
    },
    {
        "pair_sampling_scheme": "gap1_stratified_cap",
        "source": "generate_from_train_df",
        "pair_min_diff": 1,
        "min_pairs_per_user": train_min_pairs_per_user,
        "max_pairs_per_user": train_max_pairs_per_user,
        "sampling_strategy": "stratified_cap",
        "description": "Weaker preference signal: includes one-star differences.",
    },
    {
        "pair_sampling_scheme": "gap3_stratified_cap",
        "source": "generate_from_train_df",
        "pair_min_diff": 3,
        "min_pairs_per_user": train_min_pairs_per_user,
        "max_pairs_per_user": train_max_pairs_per_user,
        "sampling_strategy": "stratified_cap",
        "description": "Stronger but sparser preference signal: only rating differences >= 3.",
    },
]

pair_sampling_ablation_plan_df = pd.DataFrame(pair_sampling_ablation_plan)

display(pair_sampling_ablation_plan_df)

,pair_sampling_scheme,source,pair_min_diff,min_pairs_per_user,max_pairs_per_user,sampling_strategy,description
0,current_step1_gap2_stratified_cap,existing_step1_pairs,2,10,2000,stratified_cap,Original Step 1 policy: observed lower-rated n...
1,gap2_random_cap,generate_from_train_df,2,10,2000,random_cap,"Same threshold and cap as Step 1, but random u..."
2,gap1_stratified_cap,generate_from_train_df,1,10,2000,stratified_cap,Weaker preference signal: includes one-star di...
3,gap3_stratified_cap,generate_from_train_df,3,10,2000,stratified_cap,Stronger but sparser preference signal: only r...


### 13.3 Build pair-sampling ablation datasets

In [50]:
pair_sampling_objects = {}
pair_sampling_dataset_summaries = []
pair_sampling_index_summaries = []

for spec in pair_sampling_ablation_plan:
    scheme = spec["pair_sampling_scheme"]

    print("=" * 100)
    print("Building pair sampling variant:", scheme)
    print("=" * 100)

    if spec["source"] == "existing_step1_pairs":
        variant_pairs_idx = train_pairs_idx.copy()
        variant_dataset = train_dataset

        user_pair_counts = variant_pairs_idx.groupby("user_id").size()

        variant_summary = {
            "pair_sampling_scheme": scheme,
            "pairs": len(variant_pairs_idx),
            "users_with_pairs": variant_pairs_idx["user_id"].nunique(),
            "items_in_pairs": len(
                set(variant_pairs_idx["pos_item"].unique())
                | set(variant_pairs_idx["neg_item"].unique())
            ),
            "avg_pairs_per_user": round(user_pair_counts.mean(), 2),
            "median_pairs_per_user": round(user_pair_counts.median(), 2),
            "min_pairs_per_user": int(user_pair_counts.min()),
            "max_pairs_per_user": int(user_pair_counts.max()),
            "users_removed_by_min_pairs": 0,
            "users_affected_by_cap": None,
        }

        index_summary = {
            "pair_sampling_scheme": scheme,
            "rows_before": len(variant_pairs_idx),
            "rows_after": len(variant_pairs_idx),
            "rows_removed": 0,
            "missing_user_rows": 0,
            "missing_pos_item_rows": 0,
            "missing_neg_item_rows": 0,
            "min_user_idx": int(variant_pairs_idx["user_idx"].min()),
            "max_user_idx": int(variant_pairs_idx["user_idx"].max()),
            "min_item_idx": int(
                min(
                    variant_pairs_idx["pos_item_idx"].min(),
                    variant_pairs_idx["neg_item_idx"].min(),
                )
            ),
            "max_item_idx": int(
                max(
                    variant_pairs_idx["pos_item_idx"].max(),
                    variant_pairs_idx["neg_item_idx"].max(),
                )
            ),
            "same_pos_neg_idx": int(
                (
                    variant_pairs_idx["pos_item_idx"]
                    == variant_pairs_idx["neg_item_idx"]
                ).sum()
            ),
        }

    else:
        variant_pairs, variant_user_summary = generate_pairwise_pairs_from_interactions(
            df=train_df,
            pair_min_diff=spec["pair_min_diff"],
            min_pairs_per_user=spec["min_pairs_per_user"],
            max_pairs_per_user=spec["max_pairs_per_user"],
            sampling_strategy=spec["sampling_strategy"],
            seed=pair_ablation_seed,
        )

        variant_summary = summarize_pair_sampling_variant(
            pair_df=variant_pairs,
            user_summary_df=variant_user_summary,
            variant_name=scheme,
        )

        variant_pairs_idx, index_summary = add_pair_indices(
            variant_pairs,
            user_to_idx,
            item_to_idx,
            scheme,
        )

        index_summary["pair_sampling_scheme"] = scheme

        variant_dataset = PairwisePreferenceDataset(variant_pairs_idx)

    pair_sampling_objects[scheme] = {
        "spec": spec,
        "train_pairs_idx": variant_pairs_idx,
        "train_dataset": variant_dataset,
        "pair_summary": variant_summary,
        "index_summary": index_summary,
    }

    pair_sampling_dataset_summaries.append(variant_summary)
    pair_sampling_index_summaries.append(index_summary)

pair_sampling_dataset_summary = pd.DataFrame(pair_sampling_dataset_summaries)
pair_sampling_index_summary = pd.DataFrame(pair_sampling_index_summaries)

display(pair_sampling_dataset_summary)
display(pair_sampling_index_summary)

assert pair_sampling_index_summary["rows_removed"].sum() == 0
assert pair_sampling_index_summary["missing_user_rows"].sum() == 0
assert pair_sampling_index_summary["missing_pos_item_rows"].sum() == 0
assert pair_sampling_index_summary["missing_neg_item_rows"].sum() == 0
assert pair_sampling_index_summary["same_pos_neg_idx"].sum() == 0

Building pair sampling variant: current_step1_gap2_stratified_cap
Building pair sampling variant: gap2_random_cap
Building pair sampling variant: gap1_stratified_cap
Building pair sampling variant: gap3_stratified_cap


,pair_sampling_scheme,pairs,users_with_pairs,items_in_pairs,avg_pairs_per_user,median_pairs_per_user,min_pairs_per_user,max_pairs_per_user,users_removed_by_min_pairs,users_affected_by_cap
0,current_step1_gap2_stratified_cap,5692244,4722,3250,1205.47,1277.5,18,2000,0,NaN
1,gap2_random_cap,5692244,4722,3250,1205.47,1277.5,18,2000,0,1951.0
2,gap1_stratified_cap,7456275,4722,3250,1579.05,2000.0,143,2000,0,2928.0
3,gap3_stratified_cap,3446386,4534,3250,760.12,371.5,10,2000,188,968.0


,pair_sampling_scheme,rows_before,rows_after,rows_removed,missing_user_rows,missing_pos_item_rows,missing_neg_item_rows,min_user_idx,max_user_idx,min_item_idx,max_item_idx,same_pos_neg_idx,split
0,current_step1_gap2_stratified_cap,5692244,5692244,0,0,0,0,0,4721,0,3249,0,NaN
1,gap2_random_cap,5692244,5692244,0,0,0,0,0,4721,0,3249,0,gap2_random_cap
2,gap1_stratified_cap,7456275,7456275,0,0,0,0,0,4721,0,3249,0,gap1_stratified_cap
3,gap3_stratified_cap,3446386,3446386,0,0,0,0,1,4721,0,3249,0,gap3_stratified_cap


### 13.4 Run pair-sampling / negative-sampling ablation experiments

In [51]:
pair_sampling_summaries = []
pair_sampling_histories = []

# If available, reuse the already-trained best batch-size model
# for the original Step 1 sampling scheme.
current_step1_reuse_experiment = globals().get("best_batch_experiment", None)

for spec in pair_sampling_ablation_plan:
    scheme = spec["pair_sampling_scheme"]

    print("=" * 100)
    print("Pair sampling ablation:", scheme)
    print("=" * 100)

    variant_object = pair_sampling_objects[scheme]

    config = current_best_config.copy()

    config["pair_sampling_scheme"] = scheme
    config["pair_min_diff"] = int(spec["pair_min_diff"])
    config["min_pairs_per_user"] = int(spec["min_pairs_per_user"])
    config["max_pairs_per_user"] = int(spec["max_pairs_per_user"])
    config["sampling_strategy"] = spec["sampling_strategy"]

    config["experiment"] = (
        f"pair_sampling_"
        f"{scheme}_"
        f"dim{config['embedding_dim']}_"
        f"lr{format_float_for_name(config['learning_rate'])}_"
        f"reg{format_float_for_name(config['reg_lambda'])}_"
        f"bs{config['batch_size']}_"
        f"{'bias' if config['use_item_bias'] else 'nobias'}"
    )

    should_reuse_current = (
        scheme == "current_step1_gap2_stratified_cap"
        and current_step1_reuse_experiment is not None
        and current_step1_reuse_experiment in experiment_results
    )

    if should_reuse_current:
        print("Reusing existing best batch-size experiment:", current_step1_reuse_experiment)

        reused_result = experiment_results[current_step1_reuse_experiment]

        model = reused_result["model"]

        history = reused_result["history"].copy()
        history["stage"] = "pair_sampling_ablation_reused"
        history["pair_sampling_scheme"] = scheme
        history["reused_from_experiment"] = current_step1_reuse_experiment

        summary = reused_result["summary"].copy()
        summary["experiment"] = config["experiment"]
        summary["stage"] = "pair_sampling_ablation_reused"
        summary["pair_sampling_scheme"] = scheme
        summary["pair_min_diff"] = int(spec["pair_min_diff"])
        summary["sampling_strategy"] = spec["sampling_strategy"]
        summary["reused_from_experiment"] = current_step1_reuse_experiment

        experiment_results[config["experiment"]] = {
            "model": model,
            "history": history,
            "summary": summary,
            "stage": "pair_sampling_ablation_reused",
            "reused_from_experiment": current_step1_reuse_experiment,
        }

    else:
        model, history, summary = train_bpr_experiment(
            config=config,
            train_dataset=variant_object["train_dataset"],
            val_dataset=val_dataset,
            test_dataset=test_dataset,
            train_pairs_idx=variant_object["train_pairs_idx"],
            val_pairs_idx=val_pairs_idx,
            n_users=n_users,
            n_items=n_items,
            device=device,
            seed=random_seed,
            max_epochs=20,
            patience=3,
        )

        history = history.copy()
        history["stage"] = "pair_sampling_ablation"
        history["pair_sampling_scheme"] = scheme

        summary = summary.copy()
        summary["stage"] = "pair_sampling_ablation"
        summary["pair_sampling_scheme"] = scheme
        summary["pair_min_diff"] = int(spec["pair_min_diff"])
        summary["sampling_strategy"] = spec["sampling_strategy"]

        experiment_results[config["experiment"]] = {
            "model": model,
            "history": history,
            "summary": summary,
            "stage": "pair_sampling_ablation",
        }

    summary["train_pairs_used"] = len(variant_object["train_pairs_idx"])
    summary["train_users_used"] = variant_object["train_pairs_idx"]["user_id"].nunique()
    summary["train_items_used"] = len(
        set(variant_object["train_pairs_idx"]["pos_item"].unique())
        | set(variant_object["train_pairs_idx"]["neg_item"].unique())
    )

    pair_sampling_histories.append(history)
    pair_sampling_summaries.append(summary)

Pair sampling ablation: current_step1_gap2_stratified_cap
Reusing existing best batch-size experiment: bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias
Pair sampling ablation: gap2_random_cap


pair_sampling_gap2_random_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 01 | train_loss=0.5691 | val_macro_acc=0.7120 | val_auc=0.8526 | time=225.1s


pair_sampling_gap2_random_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 02 | train_loss=0.3945 | val_macro_acc=0.7367 | val_auc=0.8751 | time=199.0s


pair_sampling_gap2_random_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 03 | train_loss=0.3323 | val_macro_acc=0.7503 | val_auc=0.8860 | time=205.2s


pair_sampling_gap2_random_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 04 | train_loss=0.2830 | val_macro_acc=0.7567 | val_auc=0.8919 | time=213.9s


...

pair_sampling_gap3_stratified_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 09 | train_loss=0.0905 | val_macro_acc=0.7426 | val_auc=0.8879 | time=128.1s


pair_sampling_gap3_stratified_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 10 | train_loss=0.0766 | val_macro_acc=0.7430 | val_auc=0.8884 | time=129.2s


pair_sampling_gap3_stratified_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 11 | train_loss=0.0652 | val_macro_acc=0.7424 | val_auc=0.8886 | time=131.9s


pair_sampling_gap3_stratified_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 12 | train_loss=0.0558 | val_macro_acc=0.7419 | val_auc=0.8886 | time=131.1s


pair_sampling_gap3_stratified_cap_dim64_lr3em04_reg1em04_bs8192_bias | epoch 13 | train_loss=0.0481 | val_macro_acc=0.7417 | val_auc=0.8883 | time=127.0s
Early stopping at epoch 13. Best epoch: 10, best val macro accuracy: 0.7430


### 13.5 Compare pair-sampling / negative-sampling results

In [52]:
all_summaries_so_far = pd.DataFrame(
    [
        result["summary"]
        for result in experiment_results.values()
    ]
)

pair_sampling_comparison_cols = [
    "stage",
    "experiment",
    "pair_sampling_scheme",
    "pair_min_diff",
    "sampling_strategy",
    "best_epoch",
    "embedding_dim",
    "learning_rate",
    "reg_lambda",
    "batch_size",
    "use_item_bias",
    "train_pairs_used",
    "train_users_used",
    "train_items_used",
    "train_macro_accuracy_best",
    "validation_macro_accuracy_best",
    "validation_micro_accuracy_best",
    "validation_auc_best",
    "validation_log_loss_best",
    "validation_brier_best",
    "total_time_sec",
]

available_pair_sampling_cols = [
    col
    for col in pair_sampling_comparison_cols
    if col in all_summaries_so_far.columns
]

pair_sampling_comparison = (
    all_summaries_so_far[
        all_summaries_so_far["experiment"].isin(
            [
                result["summary"]["experiment"]
                for result in experiment_results.values()
                if result["summary"].get("pair_sampling_scheme") in [
                    spec["pair_sampling_scheme"]
                    for spec in pair_sampling_ablation_plan
                ]
            ]
        )
    ][available_pair_sampling_cols]
    .copy()
)

pair_sampling_comparison["macro_generalization_gap"] = (
    pair_sampling_comparison["train_macro_accuracy_best"]
    - pair_sampling_comparison["validation_macro_accuracy_best"]
)

pair_sampling_comparison = (
    pair_sampling_comparison
    .sort_values(
        [
            "validation_macro_accuracy_best",
            "validation_auc_best",
            "total_time_sec",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

display(pair_sampling_comparison.round(6))

,stage,experiment,pair_sampling_scheme,pair_min_diff,sampling_strategy,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,...,train_users_used,train_items_used,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,macro_generalization_gap
0,pair_sampling_ablation,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,gap2_random_cap,2.0,random_cap,6,64,0.0003,0.0001,8192,...,4722.0,3250.0,0.948651,0.760839,0.809004,0.896833,0.402563,0.130652,2000.85,0.187812
1,pair_sampling_ablation_reused,pair_sampling_current_step1_gap2_stratified_ca...,current_step1_gap2_stratified_cap,2.0,stratified_cap,6,64,0.0003,0.0001,8192,...,4722.0,3250.0,0.948321,0.757608,0.808433,0.896024,0.404075,0.131186,1887.21,0.190713
2,pair_sampling_ablation,pair_sampling_gap1_stratified_cap_dim64_lr3em0...,gap1_stratified_cap,1.0,stratified_cap,4,64,0.0003,0.0001,8192,...,4722.0,3250.0,0.855678,0.752951,0.801798,0.890177,0.425653,0.137963,2028.47,0.102727
3,pair_sampling_ablation,pair_sampling_gap3_stratified_cap_dim64_lr3em0...,gap3_stratified_cap,3.0,stratified_cap,10,64,0.0003,0.0001,8192,...,4534.0,3250.0,0.987667,0.742960,0.800681,0.888409,0.436638,0.139243,1695.14,0.244707


### 13.6 Select best pair-sampling scheme

In [53]:
# Primary rule:
# choose the best validation macro accuracy.
#
# Practical tie rule:
# if several schemes are within 0.001 validation macro accuracy,
# prefer the original Step 1 policy because it is more interpretable,
# keeps strong preferences, and was already justified in Step 1.

pair_sampling_macro_tolerance = 0.001

best_pair_sampling_macro = pair_sampling_comparison[
    "validation_macro_accuracy_best"
].max()

pair_sampling_candidates = (
    pair_sampling_comparison[
        pair_sampling_comparison["validation_macro_accuracy_best"]
        >= best_pair_sampling_macro - pair_sampling_macro_tolerance
    ]
    .copy()
)

pair_sampling_preference_order = {
    "current_step1_gap2_stratified_cap": 0,
    "gap2_random_cap": 1,
    "gap3_stratified_cap": 2,
    "gap1_stratified_cap": 3,
}

pair_sampling_candidates["preference_rank"] = (
    pair_sampling_candidates["pair_sampling_scheme"]
    .map(pair_sampling_preference_order)
    .fillna(99)
)

pair_sampling_candidates = (
    pair_sampling_candidates
    .sort_values(
        [
            "preference_rank",
            "validation_macro_accuracy_best",
            "validation_auc_best",
            "total_time_sec",
        ],
        ascending=[True, False, False, True],
    )
    .reset_index(drop=True)
)

best_pair_sampling_row = pair_sampling_candidates.iloc[0]

best_pair_sampling_experiment = best_pair_sampling_row["experiment"]
best_pair_sampling_scheme = best_pair_sampling_row["pair_sampling_scheme"]

selected_pair_sampling_object = pair_sampling_objects[best_pair_sampling_scheme]

selected_train_pairs_idx = selected_pair_sampling_object["train_pairs_idx"]
selected_train_dataset = selected_pair_sampling_object["train_dataset"]

current_best_config = current_best_config.copy()

current_best_config["pair_sampling_scheme"] = best_pair_sampling_scheme
current_best_config["pair_min_diff"] = int(best_pair_sampling_row["pair_min_diff"])
current_best_config["sampling_strategy"] = best_pair_sampling_row["sampling_strategy"]

current_best_config["experiment"] = (
    f"current_best_after_pair_sampling_"
    f"{best_pair_sampling_scheme}_"
    f"dim{current_best_config['embedding_dim']}_"
    f"lr{format_float_for_name(current_best_config['learning_rate'])}_"
    f"reg{format_float_for_name(current_best_config['reg_lambda'])}_"
    f"bs{current_best_config['batch_size']}_"
    f"{'bias' if current_best_config['use_item_bias'] else 'nobias'}"
)

print("Best experiment after pair-sampling ablation:", best_pair_sampling_experiment)
print("Best pair sampling scheme:", best_pair_sampling_scheme)
print("Best pair_min_diff:", current_best_config["pair_min_diff"])
print("Best sampling strategy:", current_best_config["sampling_strategy"])
print("Best validation macro accuracy:", best_pair_sampling_row["validation_macro_accuracy_best"])
print("Best validation AUC:", best_pair_sampling_row["validation_auc_best"])
print("Macro generalization gap:", best_pair_sampling_row["macro_generalization_gap"])

display(pair_sampling_candidates.round(6))

current_best_config_summary = pd.DataFrame([
    {
        "parameter": "embedding_dim",
        "value": current_best_config["embedding_dim"],
    },
    {
        "parameter": "learning_rate",
        "value": current_best_config["learning_rate"],
    },
    {
        "parameter": "reg_lambda",
        "value": current_best_config["reg_lambda"],
    },
    {
        "parameter": "batch_size",
        "value": current_best_config["batch_size"],
    },
    {
        "parameter": "use_item_bias",
        "value": current_best_config["use_item_bias"],
    },
    {
        "parameter": "pair_sampling_scheme",
        "value": current_best_config["pair_sampling_scheme"],
    },
    {
        "parameter": "pair_min_diff",
        "value": current_best_config["pair_min_diff"],
    },
    {
        "parameter": "sampling_strategy",
        "value": current_best_config["sampling_strategy"],
    },
])

display(current_best_config_summary)

Best experiment after pair-sampling ablation: pair_sampling_gap2_random_cap_dim64_lr3em04_reg1em04_bs8192_bias
Best pair sampling scheme: gap2_random_cap
Best pair_min_diff: 2
Best sampling strategy: random_cap
Best validation macro accuracy: 0.7608390566674916
Best validation AUC: 0.8968331907610073
Macro generalization gap: 0.18781209403148857


,stage,experiment,pair_sampling_scheme,pair_min_diff,sampling_strategy,best_epoch,embedding_dim,learning_rate,reg_lambda,batch_size,...,train_items_used,train_macro_accuracy_best,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,macro_generalization_gap,preference_rank
0,pair_sampling_ablation,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,gap2_random_cap,2.0,random_cap,6,64,0.0003,0.0001,8192,...,3250.0,0.948651,0.760839,0.809004,0.896833,0.402563,0.130652,2000.85,0.187812,1


,parameter,value
0,embedding_dim,64
1,learning_rate,0.0003
2,reg_lambda,0.0001
3,batch_size,8192
4,use_item_bias,True
5,pair_sampling_scheme,gap2_random_cap
6,pair_min_diff,2
7,sampling_strategy,random_cap


# 14 Final BPR Model Evaluation

In [54]:
# The final model is the best model selected after all validation-based sweeps.
# Test data is evaluated only now, after all hyperparameter choices are fixed.

final_experiment_name = best_pair_sampling_experiment
final_result = experiment_results[final_experiment_name]

final_model = final_result["model"]
final_history = final_result["history"].copy()
final_training_summary = final_result["summary"].copy()

final_model.eval()

print("Final selected experiment:", final_experiment_name)
print("Final pair sampling scheme:", current_best_config["pair_sampling_scheme"])
print("Final embedding_dim:", current_best_config["embedding_dim"])
print("Final learning_rate:", current_best_config["learning_rate"])
print("Final reg_lambda:", current_best_config["reg_lambda"])
print("Final batch_size:", current_best_config["batch_size"])
print("Final use_item_bias:", current_best_config["use_item_bias"])

Final selected experiment: pair_sampling_gap2_random_cap_dim64_lr3em04_reg1em04_bs8192_bias
Final pair sampling scheme: gap2_random_cap
Final embedding_dim: 64
Final learning_rate: 0.0003
Final reg_lambda: 0.0001
Final batch_size: 8192
Final use_item_bias: True


### 14.1 Final pairwise evaluation on train / validation / test

In [55]:
final_pairwise_eval = pd.DataFrame([
    evaluate_pairwise(
        final_model,
        selected_train_pairs_idx,
        device,
        "train_selected_pairs",
    ),
    evaluate_pairwise(
        final_model,
        val_pairs_idx,
        device,
        "validation_pairs",
    ),
    evaluate_pairwise(
        final_model,
        test_pairs_idx,
        device,
        "test_pairs",
    ),
])

display(final_pairwise_eval.round(6))

,split,pairs,users,comparison_accuracy_micro,comparison_accuracy_macro_user,roc_auc_symmetric,brier_positive_pairs,log_loss_positive_pairs,mean_score_diff,median_score_diff
0,train_selected_pairs,5692244,4722,0.939796,0.948651,0.985389,0.050559,0.184090,2.902715,2.688258
1,validation_pairs,478098,4158,0.809004,0.760839,0.896833,0.130652,0.402563,1.913548,1.567558
2,test_pairs,507550,4164,0.818034,0.774218,0.903393,0.125943,0.393546,2.040658,1.731171


### 14.2 Final generalization summary

In [56]:
final_train_macro = final_pairwise_eval.loc[
    final_pairwise_eval["split"].eq("train_selected_pairs"),
    "comparison_accuracy_macro_user",
].iloc[0]

final_val_macro = final_pairwise_eval.loc[
    final_pairwise_eval["split"].eq("validation_pairs"),
    "comparison_accuracy_macro_user",
].iloc[0]

final_test_macro = final_pairwise_eval.loc[
    final_pairwise_eval["split"].eq("test_pairs"),
    "comparison_accuracy_macro_user",
].iloc[0]

final_val_auc = final_pairwise_eval.loc[
    final_pairwise_eval["split"].eq("validation_pairs"),
    "roc_auc_symmetric",
].iloc[0]

final_test_auc = final_pairwise_eval.loc[
    final_pairwise_eval["split"].eq("test_pairs"),
    "roc_auc_symmetric",
].iloc[0]

final_generalization_summary = pd.DataFrame([
    {
        "metric": "train_macro_accuracy",
        "value": final_train_macro,
    },
    {
        "metric": "validation_macro_accuracy",
        "value": final_val_macro,
    },
    {
        "metric": "test_macro_accuracy",
        "value": final_test_macro,
    },
    {
        "metric": "train_minus_validation_macro_gap",
        "value": final_train_macro - final_val_macro,
    },
    {
        "metric": "validation_minus_test_macro_gap",
        "value": final_val_macro - final_test_macro,
    },
    {
        "metric": "validation_auc",
        "value": final_val_auc,
    },
    {
        "metric": "test_auc",
        "value": final_test_auc,
    },
])

display(final_generalization_summary.round(6))

,metric,value
0,train_macro_accuracy,0.948651
1,validation_macro_accuracy,0.760839
2,test_macro_accuracy,0.774218
3,train_minus_validation_macro_gap,0.187812
4,validation_minus_test_macro_gap,-0.013379
5,validation_auc,0.896833
6,test_auc,0.903393


### 14.3 Final training history

In [57]:
display(final_history.round(6))

,experiment,epoch,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,epoch_time_sec,train_loss,train_ranking_loss,train_l2_penalty,val_comparison_accuracy_micro,val_comparison_accuracy_macro_user,val_roc_auc_symmetric,val_log_loss_positive_pairs,val_brier_positive_pairs,val_mean_score_diff,val_median_score_diff,stage,pair_sampling_scheme
0,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,1,64,0.0003,0.0001,8192,True,225.08,0.569089,0.568956,1.323549,0.763421,0.712027,0.852573,0.482166,0.159456,0.853740,0.689499,pair_sampling_ablation,gap2_random_cap
1,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,2,64,0.0003,0.0001,8192,True,198.99,0.394547,0.394043,5.043349,0.785385,0.736703,0.875115,0.438574,0.144425,1.364415,1.079373,pair_sampling_ablation,gap2_random_cap
2,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,3,64,0.0003,0.0001,8192,True,205.20,0.332254,0.331470,7.831872,0.796820,0.750262,0.886037,0.420564,0.137663,1.589466,1.260709,pair_sampling_ablation,gap2_random_cap
3,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,4,64,0.0003,0.0001,8192,True,213.91,0.282993,0.281930,10.627908,0.803329,0.756750,0.891857,0.410755,0.133909,1.728764,1.385392,pair_sampling_ablation,gap2_random_cap
4,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,5,64,0.0003,0.0001,8192,True,251.43,0.239981,0.238622,13.589314,0.806862,0.759221,0.895173,0.405122,0.131727,1.830882,1.482554,pair_sampling_ablation,gap2_random_cap
5,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,6,64,0.0003,0.0001,8192,True,221.52,0.203115,0.201445,16.705796,0.809004,0.760839,0.896833,0.402563,0.130652,1.913548,1.567558,pair_sampling_ablation,gap2_random_cap
6,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,7,64,0.0003,0.0001,8192,True,214.91,0.172670,0.170681,19.890192,0.809754,0.760206,0.897345,0.402426,0.130400,1.986403,1.642879,pair_sampling_ablation,gap2_random_cap
7,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,8,64,0.0003,0.0001,8192,True,241.97,0.148026,0.145719,23.070402,0.809957,0.759353,0.897019,0.404344,0.130787,2.052852,1.718264,pair_sampling_ablation,gap2_random_cap
8,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,9,64,0.0003,0.0001,8192,True,227.78,0.128179,0.125560,26.186384,0.809550,0.758283,0.896172,0.407813,0.131612,2.116946,1.788774,pair_sampling_ablation,gap2_random_cap


# 15 Export Final BPR Embeddings

In [58]:
# Export the final learned BPR embeddings.
# p_u = user embeddings
# q_i = item embeddings
#
# These files will be used later for DBSCAN and downstream classifiers.

step2_results_dir = "./results_step2"
os.makedirs(step2_results_dir, exist_ok=True)

final_model.eval()

with torch.no_grad():
    user_embedding_matrix = (
        final_model.user_embedding.weight
        .detach()
        .cpu()
        .numpy()
    )

    item_embedding_matrix = (
        final_model.item_embedding.weight
        .detach()
        .cpu()
        .numpy()
    )

    if final_model.item_bias is not None:
        item_bias_vector = (
            final_model.item_bias.weight
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )
    else:
        item_bias_vector = np.zeros(n_items)


print("User embedding matrix shape:", user_embedding_matrix.shape)
print("Item embedding matrix shape:", item_embedding_matrix.shape)
print("Item bias vector shape:", item_bias_vector.shape)

assert user_embedding_matrix.shape == (n_users, current_best_config["embedding_dim"])
assert item_embedding_matrix.shape == (n_items, current_best_config["embedding_dim"])
assert item_bias_vector.shape[0] == n_items

User embedding matrix shape: (4722, 64)
Item embedding matrix shape: (3250, 64)
Item bias vector shape: (3250,)


### 15.1 Create embedding tables with original IDs

In [59]:
user_embedding_columns = [
    f"p_{dim_idx:03d}"
    for dim_idx in range(user_embedding_matrix.shape[1])
]

item_embedding_columns = [
    f"q_{dim_idx:03d}"
    for dim_idx in range(item_embedding_matrix.shape[1])
]


user_embeddings_df = pd.DataFrame(
    user_embedding_matrix,
    columns=user_embedding_columns,
)

user_embeddings_df.insert(
    0,
    "user_idx",
    range(n_users),
)

user_embeddings_df.insert(
    1,
    "user_id",
    user_embeddings_df["user_idx"].map(idx_to_user),
)


item_embeddings_df = pd.DataFrame(
    item_embedding_matrix,
    columns=item_embedding_columns,
)

item_embeddings_df.insert(
    0,
    "item_idx",
    range(n_items),
)

item_embeddings_df.insert(
    1,
    "movie_id",
    item_embeddings_df["item_idx"].map(idx_to_item),
)

item_embeddings_df["item_bias"] = item_bias_vector


display(user_embeddings_df.head())
display(item_embeddings_df.head())

,user_idx,user_id,p_000,p_001,p_002,p_003,p_004,p_005,p_006,p_007,...,p_054,p_055,p_056,p_057,p_058,p_059,p_060,p_061,p_062,p_063
0,0,1,0.027299,0.057625,0.144256,0.102114,-0.018128,-0.038718,0.279057,0.033346,...,-0.179049,0.241795,0.048778,-0.048089,-0.109588,-0.034794,-0.135660,-0.066999,-0.131665,-0.221725
1,1,2,0.048972,0.150725,0.603860,-0.305857,0.166073,-0.091961,0.461051,-0.019598,...,-0.454297,-0.106218,-0.140426,-0.042493,-0.126272,-0.291553,-0.101054,0.060885,-0.407538,0.291335
2,2,3,0.205603,-0.183252,0.022395,0.161742,0.202512,-0.076164,0.002557,0.114474,...,-0.028261,0.029026,-0.205796,0.205968,-0.443939,0.139663,-0.068764,-0.012766,-0.016416,0.125781
3,3,5,-0.183985,-0.368161,-0.013996,0.291211,0.168328,0.243547,0.332781,-0.366503,...,-0.140215,-0.285031,-0.246976,0.507154,0.365648,-0.020444,0.518965,0.324661,0.118329,0.011100
4,4,6,0.133437,-0.033939,0.248657,0.094411,-0.055039,-0.081491,0.485570,0.137255,...,0.059742,-0.231401,-0.166403,0.018796,0.066386,0.396772,-0.197218,-0.126738,-0.068290,-0.170410


,item_idx,movie_id,q_000,q_001,q_002,q_003,q_004,q_005,q_006,q_007,...,q_055,q_056,q_057,q_058,q_059,q_060,q_061,q_062,q_063,item_bias
0,0,1,-0.364326,-0.174609,0.517413,0.497861,-0.196241,0.194520,0.661545,-0.435840,...,0.355879,-0.533006,0.161613,0.305616,-0.831306,-0.203000,-0.013375,-0.112982,-0.360459,0.172780
1,1,2,0.450191,-0.074258,-0.146192,-0.213577,-0.202869,-0.317204,0.183974,0.013684,...,0.231309,0.087939,-0.073303,0.133290,-0.119906,-0.369182,-0.236907,0.288408,0.014382,-0.040055
2,2,3,0.023690,0.157482,-0.095730,0.021477,0.010826,-0.630187,-0.247355,0.105864,...,0.216113,0.313541,-0.372834,0.008202,0.070186,-0.535461,0.042750,0.081155,-0.252906,-0.077478
3,3,4,-0.049183,0.349406,-0.328853,-0.254158,-0.408783,-0.133937,-0.185500,0.021213,...,-0.443057,0.119837,-0.301804,0.117669,0.105536,0.114468,-0.101350,0.349618,-0.082149,-0.189764
4,4,5,0.126448,0.164698,-0.022967,-0.111819,-0.007676,-0.347648,0.210288,0.156055,...,0.218466,0.030304,-0.271644,0.238003,-0.055722,-0.339796,-0.042640,0.059813,-0.279468,-0.027863


### 15.2 Add metadata for interpretation

In [61]:
# Add user and movie metadata if available.
# This is useful later for cluster interpretation, fairness analysis, and reporting.
#
# Important:
# We avoid using the variable names `users` and `movies` directly here,
# because they may have been overwritten by tensors during PyTorch checks.

user_embeddings_with_metadata = user_embeddings_df.copy()
item_embeddings_with_metadata = item_embeddings_df.copy()


# Try to load user metadata safely.
users_metadata = None

if os.path.exists(f"{data_dir}/users.parquet"):
    users_metadata = pd.read_parquet(f"{data_dir}/users.parquet")
elif "users" in globals() and isinstance(globals()["users"], pd.DataFrame):
    users_metadata = globals()["users"].copy()


# Try to load movie metadata safely.
movies_metadata = None

if os.path.exists(f"{data_dir}/movies.parquet"):
    movies_metadata = pd.read_parquet(f"{data_dir}/movies.parquet")
elif "movies" in globals() and isinstance(globals()["movies"], pd.DataFrame):
    movies_metadata = globals()["movies"].copy()


if users_metadata is not None:
    if "user_id" not in users_metadata.columns:
        raise ValueError("users metadata exists, but it does not contain 'user_id'.")

    user_embeddings_with_metadata = user_embeddings_with_metadata.merge(
        users_metadata,
        on="user_id",
        how="left",
    )

    print("User metadata merged.")
else:
    print("User metadata was not found. Saving user embeddings without metadata.")


if movies_metadata is not None:
    if "movie_id" not in movies_metadata.columns:
        raise ValueError("movies metadata exists, but it does not contain 'movie_id'.")

    item_embeddings_with_metadata = item_embeddings_with_metadata.merge(
        movies_metadata,
        on="movie_id",
        how="left",
    )

    print("Movie metadata merged.")
else:
    print("Movie metadata was not found. Saving item embeddings without metadata.")


metadata_merge_summary = pd.DataFrame([
    {
        "table": "user_embeddings_with_metadata",
        "rows": len(user_embeddings_with_metadata),
        "columns": user_embeddings_with_metadata.shape[1],
        "missing_metadata_rows": (
            user_embeddings_with_metadata.isna().any(axis=1).sum()
            if users_metadata is not None
            else None
        ),
    },
    {
        "table": "item_embeddings_with_metadata",
        "rows": len(item_embeddings_with_metadata),
        "columns": item_embeddings_with_metadata.shape[1],
        "missing_metadata_rows": (
            item_embeddings_with_metadata.isna().any(axis=1).sum()
            if movies_metadata is not None
            else None
        ),
    },
])

display(metadata_merge_summary)
display(user_embeddings_with_metadata.head())
display(item_embeddings_with_metadata.head())

User metadata merged.
Movie metadata merged.


,table,rows,columns,missing_metadata_rows
0,user_embeddings_with_metadata,4722,70,0
1,item_embeddings_with_metadata,3250,69,0


,user_idx,user_id,p_000,p_001,p_002,p_003,p_004,p_005,p_006,p_007,...,p_058,p_059,p_060,p_061,p_062,p_063,gender,age,occupation,zip_code
0,0,1,0.027299,0.057625,0.144256,0.102114,-0.018128,-0.038718,0.279057,0.033346,...,-0.109588,-0.034794,-0.135660,-0.066999,-0.131665,-0.221725,F,1,10,48067
1,1,2,0.048972,0.150725,0.603860,-0.305857,0.166073,-0.091961,0.461051,-0.019598,...,-0.126272,-0.291553,-0.101054,0.060885,-0.407538,0.291335,M,56,16,70072
2,2,3,0.205603,-0.183252,0.022395,0.161742,0.202512,-0.076164,0.002557,0.114474,...,-0.443939,0.139663,-0.068764,-0.012766,-0.016416,0.125781,M,25,15,55117
3,3,5,-0.183985,-0.368161,-0.013996,0.291211,0.168328,0.243547,0.332781,-0.366503,...,0.365648,-0.020444,0.518965,0.324661,0.118329,0.011100,M,25,20,55455
4,4,6,0.133437,-0.033939,0.248657,0.094411,-0.055039,-0.081491,0.485570,0.137255,...,0.066386,0.396772,-0.197218,-0.126738,-0.068290,-0.170410,F,50,9,55117


,item_idx,movie_id,q_000,q_001,q_002,q_003,q_004,q_005,q_006,q_007,...,q_057,q_058,q_059,q_060,q_061,q_062,q_063,item_bias,title,genres
0,0,1,-0.364326,-0.174609,0.517413,0.497861,-0.196241,0.194520,0.661545,-0.435840,...,0.161613,0.305616,-0.831306,-0.203000,-0.013375,-0.112982,-0.360459,0.172780,Toy Story (1995),Animation|Children's|Comedy
1,1,2,0.450191,-0.074258,-0.146192,-0.213577,-0.202869,-0.317204,0.183974,0.013684,...,-0.073303,0.133290,-0.119906,-0.369182,-0.236907,0.288408,0.014382,-0.040055,Jumanji (1995),Adventure|Children's|Fantasy
2,2,3,0.023690,0.157482,-0.095730,0.021477,0.010826,-0.630187,-0.247355,0.105864,...,-0.372834,0.008202,0.070186,-0.535461,0.042750,0.081155,-0.252906,-0.077478,Grumpier Old Men (1995),Comedy|Romance
3,3,4,-0.049183,0.349406,-0.328853,-0.254158,-0.408783,-0.133937,-0.185500,0.021213,...,-0.301804,0.117669,0.105536,0.114468,-0.101350,0.349618,-0.082149,-0.189764,Waiting to Exhale (1995),Comedy|Drama
4,4,5,0.126448,0.164698,-0.022967,-0.111819,-0.007676,-0.347648,0.210288,0.156055,...,-0.271644,0.238003,-0.055722,-0.339796,-0.042640,0.059813,-0.279468,-0.027863,Father of the Bride Part II (1995),Comedy


### 15.3 Save embeddings and final model artifacts

In [62]:
# Save compact NumPy arrays.

np.save(
    f"{step2_results_dir}/user_embeddings_p_u.npy",
    user_embedding_matrix,
)

np.save(
    f"{step2_results_dir}/item_embeddings_q_i.npy",
    item_embedding_matrix,
)

np.save(
    f"{step2_results_dir}/item_bias.npy",
    item_bias_vector,
)


# Save tabular versions with original IDs.

user_embeddings_df.to_parquet(
    f"{step2_results_dir}/user_embeddings_p_u.parquet",
    index=False,
)

item_embeddings_df.to_parquet(
    f"{step2_results_dir}/item_embeddings_q_i.parquet",
    index=False,
)

user_embeddings_with_metadata.to_parquet(
    f"{step2_results_dir}/user_embeddings_p_u_with_metadata.parquet",
    index=False,
)

item_embeddings_with_metadata.to_parquet(
    f"{step2_results_dir}/item_embeddings_q_i_with_metadata.parquet",
    index=False,
)


# Save mapping tables.

user_index_table.to_parquet(
    f"{step2_results_dir}/user_index_mapping.parquet",
    index=False,
)

item_index_table.to_parquet(
    f"{step2_results_dir}/item_index_mapping.parquet",
    index=False,
)


# Save final PyTorch model weights.

torch.save(
    final_model.state_dict(),
    f"{step2_results_dir}/final_bpr_model_state_dict.pt",
)


print("Saved final Step 2 embeddings and model artifacts to:", step2_results_dir)

Saved final Step 2 embeddings and model artifacts to: ./results_step2


### 15.4 Save final configuration and evaluation results

In [63]:
final_step2_config = current_best_config.copy()

final_step2_config["final_experiment_name"] = final_experiment_name
final_step2_config["selection_metric"] = "validation_macro_accuracy_best"
final_step2_config["test_used_for_selection"] = False
final_step2_config["n_users"] = int(n_users)
final_step2_config["n_items"] = int(n_items)
final_step2_config["user_embedding_file"] = "user_embeddings_p_u.npy"
final_step2_config["item_embedding_file"] = "item_embeddings_q_i.npy"


with open(f"{step2_results_dir}/final_step2_config.json", "w") as f:
    json.dump(final_step2_config, f, indent=2)


final_pairwise_eval.to_parquet(
    f"{step2_results_dir}/final_pairwise_evaluation.parquet",
    index=False,
)

final_generalization_summary.to_parquet(
    f"{step2_results_dir}/final_generalization_summary.parquet",
    index=False,
)

final_history.to_parquet(
    f"{step2_results_dir}/final_training_history.parquet",
    index=False,
)


print("Saved final Step 2 config and evaluation summaries.")

Saved final Step 2 config and evaluation summaries.


### 15.5 Export all experiment summaries

In [64]:
all_experiment_summaries = pd.DataFrame([
    result["summary"]
    for result in experiment_results.values()
])

all_experiment_summaries = all_experiment_summaries.sort_values(
    ["validation_macro_accuracy_best", "validation_auc_best"],
    ascending=False,
).reset_index(drop=True)

all_experiment_summaries.to_parquet(
    f"{step2_results_dir}/all_experiment_summaries.parquet",
    index=False,
)

display(all_experiment_summaries.round(6))

,experiment,best_epoch,best_val_macro_accuracy,total_time_sec,embedding_dim,learning_rate,reg_lambda,batch_size,use_item_bias,train_macro_accuracy_best,...,validation_log_loss_best,validation_brier_best,stage,reused_from_experiment,pair_sampling_scheme,pair_min_diff,sampling_strategy,train_pairs_used,train_users_used,train_items_used
0,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,6,0.760839,2000.85,64,0.0003,0.000100,8192,True,0.948651,...,0.402563,0.130652,pair_sampling_ablation,NaN,gap2_random_cap,2.0,random_cap,5692244.0,4722.0,3250.0
1,bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias,6,0.757608,1887.21,64,0.0003,0.000100,8192,True,0.948321,...,0.404075,0.131186,batch_size_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,pair_sampling_current_step1_gap2_stratified_ca...,6,0.757608,1887.21,64,0.0003,0.000100,8192,True,0.948321,...,0.404075,0.131186,pair_sampling_ablation_reused,bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias,current_step1_gap2_stratified_cap,2.0,stratified_cap,5692244.0,4722.0,3250.0
3,lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias,10,0.756726,2574.66,64,0.0003,0.000100,16384,True,0.957874,...,0.403846,0.131054,learning_rate_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bs_sweep_dim64_lr3em04_reg1em04_bs16384_bias,10,0.756726,2574.66,64,0.0003,0.000100,16384,True,0.957874,...,0.403846,0.131054,batch_size_sweep_reused,lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias,NaN,NaN,NaN,NaN,NaN,NaN
5,bs_sweep_dim64_lr3em04_reg1em04_bs32768_bias,14,0.756656,3403.24,64,0.0003,0.000100,32768,True,0.949681,...,0.404601,0.131510,batch_size_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias,20,0.755769,3635.83,64,0.0003,0.000100,65536,True,0.937226,...,0.407325,0.132552,batch_size_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,pair_sampling_gap1_stratified_cap_dim64_lr3em0...,4,0.752951,2028.47,64,0.0003,0.000100,8192,True,0.855678,...,0.425653,0.137963,pair_sampling_ablation,NaN,gap1_stratified_cap,1.0,stratified_cap,7456275.0,4722.0,3250.0
8,dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias,2,0.752302,968.92,64,0.0010,0.000100,16384,True,0.931816,...,0.409997,0.133594,embedding_dim_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias,3,0.751572,1072.08,32,0.0010,0.000100,16384,True,0.919164,...,0.412634,0.134518,regularization_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 16 Step 2 final audit and report tables

In [65]:
# Final Step 2 audit and report-ready tables.
# This block checks that all required Step 2 outputs exist and prepares compact
# summary tables for the written report.

from pathlib import Path

step2_results_path = Path(step2_results_dir)

required_step2_files = [
    "user_embeddings_p_u.npy",
    "item_embeddings_q_i.npy",
    "item_bias.npy",
    "user_embeddings_p_u.parquet",
    "item_embeddings_q_i.parquet",
    "user_embeddings_p_u_with_metadata.parquet",
    "item_embeddings_q_i_with_metadata.parquet",
    "user_index_mapping.parquet",
    "item_index_mapping.parquet",
    "final_bpr_model_state_dict.pt",
    "final_step2_config.json",
    "final_pairwise_evaluation.parquet",
    "final_generalization_summary.parquet",
    "final_training_history.parquet",
    "all_experiment_summaries.parquet",
]

artifact_audit = pd.DataFrame([
    {
        "file": filename,
        "exists": (step2_results_path / filename).exists(),
        "size_mb": (
            round((step2_results_path / filename).stat().st_size / (1024 ** 2), 4)
            if (step2_results_path / filename).exists()
            else None
        ),
    }
    for filename in required_step2_files
])

display(artifact_audit)

assert artifact_audit["exists"].all(), "Some required Step 2 artifacts are missing."

,file,exists,size_mb
0,user_embeddings_p_u.npy,True,1.1530
1,item_embeddings_q_i.npy,True,0.7936
2,item_bias.npy,True,0.0125
3,user_embeddings_p_u.parquet,True,1.7067
4,item_embeddings_q_i.parquet,True,1.1773
5,user_embeddings_p_u_with_metadata.parquet,True,1.7355
6,item_embeddings_q_i_with_metadata.parquet,True,1.2421
7,user_index_mapping.parquet,True,0.0525
8,item_index_mapping.parquet,True,0.0359
9,final_bpr_model_state_dict.pt,True,1.9611


### 16.1 Final selected configuration summary

In [66]:
final_selected_config_summary = pd.DataFrame([
    {
        "parameter": "model",
        "selected_value": "BPR matrix factorization",
        "reason": "Pairwise objective directly optimizes preferred-item-over-less-preferred-item ranking.",
    },
    {
        "parameter": "embedding_dim",
        "selected_value": final_step2_config["embedding_dim"],
        "reason": "Selected using validation macro pairwise accuracy during embedding-dimension sweep.",
    },
    {
        "parameter": "learning_rate",
        "selected_value": final_step2_config["learning_rate"],
        "reason": "Selected using validation macro pairwise accuracy during learning-rate sweep.",
    },
    {
        "parameter": "reg_lambda",
        "selected_value": final_step2_config["reg_lambda"],
        "reason": "Selected using validation macro pairwise accuracy during regularization sweep.",
    },
    {
        "parameter": "batch_size",
        "selected_value": final_step2_config["batch_size"],
        "reason": "Selected as the fastest batch size within tolerance of the best validation macro accuracy.",
    },
    {
        "parameter": "use_item_bias",
        "selected_value": final_step2_config["use_item_bias"],
        "reason": "Item bias was kept because it models item-level popularity while embeddings model user-item preference structure.",
    },
    {
        "parameter": "pair_sampling_scheme",
        "selected_value": final_step2_config.get("pair_sampling_scheme", None),
        "reason": "Selected using the pair-sampling / negative-sampling ablation on validation macro pairwise accuracy.",
    },
    {
        "parameter": "pair_min_diff",
        "selected_value": final_step2_config.get("pair_min_diff", None),
        "reason": "Controls how strong the preference signal must be when constructing item pairs.",
    },
    {
        "parameter": "selection_metric",
        "selected_value": final_step2_config["selection_metric"],
        "reason": "Validation macro pairwise accuracy is user-balanced and avoids letting heavy users dominate model selection.",
    },
    {
        "parameter": "test_used_for_selection",
        "selected_value": final_step2_config["test_used_for_selection"],
        "reason": "False. Test data was held out until final evaluation.",
    },
])

display(final_selected_config_summary)

,parameter,selected_value,reason
0,model,BPR matrix factorization,Pairwise objective directly optimizes preferre...
1,embedding_dim,64,Selected using validation macro pairwise accur...
2,learning_rate,0.0003,Selected using validation macro pairwise accur...
3,reg_lambda,0.0001,Selected using validation macro pairwise accur...
4,batch_size,8192,Selected as the fastest batch size within tole...
5,use_item_bias,True,Item bias was kept because it models item-leve...
6,pair_sampling_scheme,gap2_random_cap,Selected using the pair-sampling / negative-sa...
7,pair_min_diff,2,Controls how strong the preference signal must...
8,selection_metric,validation_macro_accuracy_best,Validation macro pairwise accuracy is user-bal...
9,test_used_for_selection,False,False. Test data was held out until final eval...


### 16.2 Final performance summary

In [67]:
final_performance_report = final_pairwise_eval.copy()

final_performance_report = final_performance_report[
    [
        "split",
        "pairs",
        "users",
        "comparison_accuracy_micro",
        "comparison_accuracy_macro_user",
        "roc_auc_symmetric",
        "brier_positive_pairs",
        "log_loss_positive_pairs",
        "mean_score_diff",
        "median_score_diff",
    ]
].copy()

display(final_performance_report.round(6))

final_generalization_report = final_generalization_summary.copy()
display(final_generalization_report.round(6))

,split,pairs,users,comparison_accuracy_micro,comparison_accuracy_macro_user,roc_auc_symmetric,brier_positive_pairs,log_loss_positive_pairs,mean_score_diff,median_score_diff
0,train_selected_pairs,5692244,4722,0.939796,0.948651,0.985389,0.050559,0.184090,2.902715,2.688258
1,validation_pairs,478098,4158,0.809004,0.760839,0.896833,0.130652,0.402563,1.913548,1.567558
2,test_pairs,507550,4164,0.818034,0.774218,0.903393,0.125943,0.393546,2.040658,1.731171


,metric,value
0,train_macro_accuracy,0.948651
1,validation_macro_accuracy,0.760839
2,test_macro_accuracy,0.774218
3,train_minus_validation_macro_gap,0.187812
4,validation_minus_test_macro_gap,-0.013379
5,validation_auc,0.896833
6,test_auc,0.903393


### 16.3 Hyperparameter impact summary

In [68]:
# Build a compact impact table from the completed sweeps.
# The impact is measured as the validation macro accuracy range within each sweep.

summary_df = all_experiment_summaries.copy()

impact_rows = []

for stage_name, parameter_name in [
    ("regularization_sweep", "reg_lambda"),
    ("embedding_dim_sweep", "embedding_dim"),
    ("learning_rate_sweep", "learning_rate"),
    ("batch_size_sweep", "batch_size"),
    ("pair_sampling_ablation", "pair_sampling_scheme"),
]:
    stage_df = summary_df[summary_df["stage"].eq(stage_name)].copy()

    if len(stage_df) == 0:
        continue

    best_row = stage_df.sort_values(
        ["validation_macro_accuracy_best", "validation_auc_best"],
        ascending=[False, False],
    ).iloc[0]

    worst_row = stage_df.sort_values(
        ["validation_macro_accuracy_best", "validation_auc_best"],
        ascending=[True, True],
    ).iloc[0]

    impact_rows.append({
        "sweep": stage_name,
        "parameter": parameter_name,
        "n_experiments": len(stage_df),
        "best_experiment": best_row["experiment"],
        "best_value": best_row.get(parameter_name, None),
        "best_validation_macro_accuracy": best_row["validation_macro_accuracy_best"],
        "worst_experiment": worst_row["experiment"],
        "worst_value": worst_row.get(parameter_name, None),
        "worst_validation_macro_accuracy": worst_row["validation_macro_accuracy_best"],
        "validation_macro_accuracy_range": (
            best_row["validation_macro_accuracy_best"]
            - worst_row["validation_macro_accuracy_best"]
        ),
    })

hyperparameter_impact_summary = (
    pd.DataFrame(impact_rows)
    .sort_values("validation_macro_accuracy_range", ascending=False)
    .reset_index(drop=True)
)

display(hyperparameter_impact_summary.round(6))

,sweep,parameter,n_experiments,best_experiment,best_value,best_validation_macro_accuracy,worst_experiment,worst_value,worst_validation_macro_accuracy,validation_macro_accuracy_range
0,pair_sampling_ablation,pair_sampling_scheme,3,pair_sampling_gap2_random_cap_dim64_lr3em04_re...,gap2_random_cap,0.760839,pair_sampling_gap3_stratified_cap_dim64_lr3em0...,gap3_stratified_cap,0.742960,0.017879
1,learning_rate_sweep,learning_rate,2,lr_sweep_dim64_lr3em04_reg1em04_bs16384_bias,0.0003,0.756726,lr_sweep_dim64_lr3em03_reg1em04_bs16384_bias,0.003,0.745033,0.011693
2,embedding_dim_sweep,embedding_dim,2,dim_sweep_dim64_lr1em03_reg1em04_bs16384_bias,64,0.752302,dim_sweep_dim16_lr1em03_reg1em04_bs16384_bias,16,0.745718,0.006584
3,batch_size_sweep,batch_size,3,bs_sweep_dim64_lr3em04_reg1em04_bs8192_bias,8192,0.757608,bs_sweep_dim64_lr3em04_reg1em04_bs65536_bias,65536,0.755769,0.001839
4,regularization_sweep,reg_lambda,4,reg_sweep_dim32_lr1em03_reg1em04_bs16384_bias,0.0001,0.751572,reg_sweep_dim32_lr1em03_reg3em05_bs16384_bias,0.00003,0.751432,0.000140


### 16.4 Pair-sampling sensitivity summary

In [69]:
pair_sampling_sensitivity_report = pair_sampling_comparison.copy()

pair_sampling_sensitivity_report = pair_sampling_sensitivity_report[
    [
        "pair_sampling_scheme",
        "pair_min_diff",
        "sampling_strategy",
        "train_pairs_used",
        "train_users_used",
        "train_items_used",
        "best_epoch",
        "validation_macro_accuracy_best",
        "validation_micro_accuracy_best",
        "validation_auc_best",
        "validation_log_loss_best",
        "validation_brier_best",
        "total_time_sec",
        "macro_generalization_gap",
    ]
].copy()

pair_sampling_sensitivity_report = pair_sampling_sensitivity_report.sort_values(
    ["validation_macro_accuracy_best", "validation_auc_best"],
    ascending=[False, False],
).reset_index(drop=True)

display(pair_sampling_sensitivity_report.round(6))

,pair_sampling_scheme,pair_min_diff,sampling_strategy,train_pairs_used,train_users_used,train_items_used,best_epoch,validation_macro_accuracy_best,validation_micro_accuracy_best,validation_auc_best,validation_log_loss_best,validation_brier_best,total_time_sec,macro_generalization_gap
0,gap2_random_cap,2.0,random_cap,5692244.0,4722.0,3250.0,6,0.760839,0.809004,0.896833,0.402563,0.130652,2000.85,0.187812
1,current_step1_gap2_stratified_cap,2.0,stratified_cap,5692244.0,4722.0,3250.0,6,0.757608,0.808433,0.896024,0.404075,0.131186,1887.21,0.190713
2,gap1_stratified_cap,1.0,stratified_cap,7456275.0,4722.0,3250.0,4,0.752951,0.801798,0.890177,0.425653,0.137963,2028.47,0.102727
3,gap3_stratified_cap,3.0,stratified_cap,3446386.0,4534.0,3250.0,10,0.742960,0.800681,0.888409,0.436638,0.139243,1695.14,0.244707
